# Điểm mù đuôi `.vn` của bộ phát hiện URL lừa đảo (XAI): làm lại từng bước và kiểm mọi con số

Notebook này làm lại toàn bộ thí nghiệm của bài từ bộ dữ liệu PhishVN đã công bố, trong một file duy nhất. Mọi hàm đều nằm ngay trong notebook,
nên đọc từ trên xuống là thấy hết cách mỗi con số được tính ra. Notebook chạy độc lập, không cần repo nào khác.

### Cách chạy

1. Tải thư mục chứa notebook này (gồm `suffix_blindspot_walkthrough.ipynb`, thư mục `expected/` và `requirements.txt`).
2. Cài thư viện đúng phiên bản: `pip install -r requirements.txt` (Python 3.12).
3. Mở notebook và chạy hết (Run All).

Lần chạy đầu, notebook tự tải gói `PhishVN_v3.1.0_open.zip` (khoảng 6 MB) từ bản 4 của bộ dữ liệu PhishVN trên Mendeley Data
(DOI 10.17632/b97hxbxtpd.4), kiểm SHA-256 của gói, rồi lấy ra hai file: `dataset_url.csv` (53.116 URL, nhãn, ngày thu thập) và `vn_compphish.csv`
(21 đặc trưng từ vựng của mỗi URL, cùng thứ tự dòng). Danh sách token thương hiệu `brand_tokens.json` đi kèm trong `expected/`.

Mọi file tải về và mọi kết quả trung gian nằm trong thư mục `p6_work/` cạnh notebook. Cả notebook chạy khoảng 40 phút trên CPU; riêng mạng CharCNN
(Bước 10, năm seed) chiếm khoảng 30 phút, các bước còn lại chỉ vài chục giây mỗi bước.

### Notebook kiểm gì

Mỗi thí nghiệm in kết quả ra màn hình và ghi lại đúng các bảng kết quả (CSV) mà bài đọc; notebook so từng byte các bảng đó với bản kỳ vọng trong
`expected/`. Notebook không vẽ hình.

Mỗi bước có phần giải thích, code, và dòng `[step] x/y checks match` ở cuối. Bước 13 gom tất cả lại. Kết quả giống hệt trên máy khác khi cài đúng
phiên bản thư viện trong `requirements.txt`; mạng CharCNN huấn luyện bằng torch ở chế độ tất định trên CPU.

### Các bước

| Bước | Làm gì |
|---|---|
| 0 | thiết lập, tải dữ liệu |
| 1 | các hàm dùng chung: nạp dữ liệu, chia theo thời gian, các họ mô hình |
| 2 | SHAP theo giao thức đánh giá: mô hình dựa vào gì khi chia ngẫu nhiên và khi chia theo thời gian |
| 3 | `tld_len` đọc gì ở đuôi `.vn`: SHAP theo từng đuôi tên miền, và token thương hiệu trong tên miền |
| 4 | ngưỡng quyết định theo nhóm (`.vn` và phần còn lại) ở cùng một mức báo nhầm |
| 5 | thâm hụt của từng tin `.vn` bị bỏ sót, tính trong không gian log-odds |
| 6 | điểm mù theo bốn tầng đuôi tên miền, rò rỉ benign, vị trí token thương hiệu |
| 7 | biên chi phí: muốn bắt `.vn` tốt hơn thì phải trả bao nhiêu báo nhầm |
| 8 | SHAP trôi theo thời gian trên luồng phishing về sau |
| 9 | các ca cụ thể |
| 10 | CharCNN đọc ký tự URL: điểm mù có còn không |
| 11 | ablation tiến cứu: bỏ `tld_len` hoặc cả họ đặc trưng độ dài rồi huấn luyện lại |
| 12 | độ nhạy theo tầng nhãn, và chạy lại trên corpus một-dòng-mỗi-URL |
| 13 | tổng hợp |

### Thuật ngữ

| Từ | Nghĩa |
|---|---|
| SHAP | cách chia điểm dự đoán của mô hình cho từng đặc trưng; giá trị dương đẩy về phía phishing, âm đẩy về phía benign |
| `tld_len` | độ dài đuôi tên miền (`.vn` là 2, `.com.vn` là 6); đặc trưng mà mô hình dựa vào nhiều nhất |
| FNR | tỷ lệ phishing bị bỏ sót; FPR là tỷ lệ benign bị báo nhầm |
| chia theo thời gian | huấn luyện trên 70% tin phishing cũ nhất, kiểm trên 30% mới nhất; bỏ khỏi tập kiểm các tên miền đã có trong tập huấn luyện |
| tên miền đăng ký (`rdom`) | phần tên miền mua được, ví dụ `abc.com.vn` của `login.abc.com.vn` |
| bốn tầng đuôi | `vn_short` (`.vn` trần), `cc_short` (đuôi quốc gia 2 ký tự khác), `vn_compound` (`.com.vn`, `.edu.vn`...), `long` (đuôi dài hơn 2 ký tự, không phải `.vn`) |
| log-odds (margin) | đầu ra thô của mô hình trước khi đổi thành xác suất; không gian mà SHAP cộng được |

## Bước 0: Thiết lập và tải dữ liệu

Các thí nghiệm đọc và ghi theo một cây thư mục cố định (`data/processed/...`). Notebook dựng đúng cây đó bên trong
`p6_work/`, đặt hai file dữ liệu vào `data/processed/`, rồi chuyển thư mục làm việc vào đó. Nhờ vậy code thí nghiệm giữ nguyên đường dẫn gốc mà
không ghi ra ngoài `p6_work/`.

Ô thứ hai định nghĩa các hàm kiểm: `check`, `report`, `check_file` (so từng byte một file với bản kỳ vọng) và `check_frame` (so hai bảng CSV theo
từng ô, dùng khi chỉ cần so các cột chung).

In [ ]:
import os, sys, io, re, csv, json, math, time, hashlib, zipfile, itertools, inspect, unicodedata, warnings, filecmp, urllib.request
from pathlib import Path
from types import SimpleNamespace
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

MENDELEY_ZIP = ("https://data.mendeley.com/public-files/datasets/b97hxbxtpd/files/"
                "d5724d4d-9c2b-46b7-810e-cf290c3bfd87/file_downloaded")
ZIP_SHA256 = "308351e9d0c0fca13a81f2f63524a7dee08ccba6c775848d125b3da5525d0ab5"
DATA_SHA256 = {"dataset_url.csv": "8e1c6325a82177014ac892a9aaf8c4d3cbbc986b47fd1ecac8899c865fb6eade",
               "vn_compphish.csv": "ce64567f4ab955b214b13a268546a460f2dd827b18432d0971b3f5a3ef06bbbf"}
EXPECTED_URL = "https://raw.githubusercontent.com/vuthainguyen1602/phishvn/master/notebooks/suffix_blindspot/expected"

def fetch(url, dest):
    # Mendeley refuses Python's default user agent
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (PhishVN walkthrough notebook)"})
    Path(dest).write_bytes(urllib.request.urlopen(req, timeout=120).read())

HOME = Path.cwd()
WORK = HOME / "p6_work"
(WORK / "data/processed/p6").mkdir(parents=True, exist_ok=True)
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()

zip_path = WORK / "PhishVN_v3.1.0_open.zip"
if not zip_path.exists():
    fetch(MENDELEY_ZIP, zip_path)
assert sha(zip_path) == ZIP_SHA256, "the downloaded package is not the published one"
with zipfile.ZipFile(zip_path) as z:
    for name in DATA_SHA256:
        (WORK / "data/processed" / name).write_bytes(z.read(f"data/{name}"))
for name, h in DATA_SHA256.items():
    assert sha(WORK / "data/processed" / name) == h, name

EXPECTED = HOME / "expected"
EXPECTED_FILES = json.loads((EXPECTED / "manifest.json").read_text()) if (EXPECTED / "manifest.json").exists() else None
if EXPECTED_FILES is None:
    EXPECTED = WORK / "expected"
    EXPECTED.mkdir(exist_ok=True)
    fetch(f"{EXPECTED_URL}/manifest.json", EXPECTED / "manifest.json")
    EXPECTED_FILES = json.loads((EXPECTED / "manifest.json").read_text())
    for f in EXPECTED_FILES:
        (EXPECTED / f).parent.mkdir(parents=True, exist_ok=True)
        if not (EXPECTED / f).exists():
            fetch(f"{EXPECTED_URL}/{f}", EXPECTED / f)
(WORK / "data/processed/brand_tokens.json").write_bytes((EXPECTED / "brand_tokens.json").read_bytes())

os.chdir(WORK)
ROOT = str(WORK)
print("work tree:", WORK, "| expected:", EXPECTED, f"({len(EXPECTED_FILES)} files)")

In [ ]:
CHECKS = []

def check(step, name, mine, ref):
    ok = mine == ref
    CHECKS.append({"step": step, "check": name, "notebook": mine, "saved": ref, "match": bool(ok)})
    return ok

def report(step):
    rows_ = [c for c in CHECKS if c["step"] == step]
    bad = [c for c in rows_ if not c["match"]]
    print(f"[{step}] {len(rows_) - len(bad)}/{len(rows_)} checks match")
    for c in bad:
        print(f"    MISMATCH {c['check']}: notebook={c['notebook']!r}  saved={c['saved']!r}")

def check_file(step, produced, expected_name=None):
    # A file this notebook wrote, byte for byte against the expected copy of the same name
    produced = Path(produced)
    ref = EXPECTED / (expected_name or produced.name)
    same = produced.exists() and filecmp.cmp(produced, ref, shallow=False)
    check(step, f"{expected_name or produced.name} byte-identical", same, True)
    if not same and produced.exists() and produced.suffix == ".csv":
        a, b = pd.read_csv(produced), pd.read_csv(ref)
        print(f"    {produced.name}: shape {a.shape} vs {b.shape}")

def check_frame(step, label, produced, expected_name, columns=None):
    # Two CSVs compared cell by cell on the given (or the shared) columns
    a, b = pd.read_csv(produced), pd.read_csv(EXPECTED / expected_name)
    cols = columns or [c for c in b.columns if c in a.columns]
    check(step, f"{label}: {len(cols)} columns equal", a[cols].equals(b[cols]), True)

## Bước 1: Các hàm dùng chung

Mọi thí nghiệm của bài dùng chung một số hàm:
- `registered_domain`: lấy tên miền đăng ký theo Public Suffix List (bản đóng gói sẵn trong `tldextract`, không tải từ mạng);
- `parse_date`: đọc ngày thu thập, thử dạng `dd/mm/yyyy` trước rồi `yyyy-mm-dd` (chỉ thử một dạng sẽ mất gần nửa corpus);
- `add_label`: đổi nhãn chữ thành 0/1, và từ chối nhãn không rõ thay vì lặng lẽ coi là benign;
- `load`: ghép bảng đặc trưng với ngày thu thập, bỏ dòng thiếu đặc trưng, thêm cột `rdom`;
- `split_phishing`: cắt tin phishing theo thời gian (70% cũ để huấn luyện, 30% mới để kiểm) và bỏ khỏi tập kiểm mọi tên miền đã có trong tập huấn luyện;
- `COMPPHISH`: 21 đặc trưng từ vựng của URL. Cột `is_https` bị loại vì trong dữ liệu benign được lưu dạng `https` còn phishing dạng `http`, nên nó
  là một bộ dự đoán hoàn hảo giả tạo;
- `make_model` và `make_any_model`: các họ mô hình với cấu hình cố định (CatBoost, XGBoost, LightGBM 300 cây; LogReg, RandomForest, HistGB, MLP;
  và mô hình xếp chồng CatBoost + LogReg);
- `wilson`: khoảng tin cậy Wilson 95% cho một tỷ lệ.

In [ ]:
import tldextract
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, balanced_accuracy_score,
                             matthews_corrcoef)

_PSL = tldextract.TLDExtract(suffix_list_urls=(), include_psl_private_domains=True)

def registered_domain(host):
    r = _PSL(host)
    new = getattr(r, "top_domain_under_public_suffix", None)
    return ((r.registered_domain if new is None else new).lower() or host)

In [ ]:
COMPPHISH = ["url_len", "dom_len", "is_ip", "tld_len", "subdom_cnt", "letter_cnt", "digit_cnt",
             "special_cnt", "eq_cnt", "qm_cnt", "amp_cnt", "dot_cnt", "dash_cnt", "under_cnt",
             "letter_ratio", "digit_ratio", "spec_ratio", "slash_cnt", "entropy",
             "path_len", "query_len"]


def add_label(df):
    # Unknown is not a negative. Respect explicit training eligibility and reject
    # unresolved labels instead of silently converting them to benign.
    if "training_eligible" in df:
        eligible = df["training_eligible"].astype(str).str.lower().isin({"1", "true"})
        if not eligible.all():
            raise ValueError("observations excluded by the selected label policy cannot be used for training/evaluation")
    lab = df["label"]
    if lab.dtype != object:
        numeric = pd.to_numeric(lab, errors="coerce")
        if not numeric.isin([0, 1]).all():
            raise ValueError("labels must be known binary outcomes, not missing or unknown")
        df["y"] = numeric.astype(int)
    else:
        pos = {"phishing", "phish", "spam", "smishing", "malicious", "1"}
        negative = {"benign", "legitimate", "ham", "0"}
        normalized = lab.astype(str).str.strip().str.lower()
        if not normalized.isin(pos | negative).all():
            raise ValueError("unknown label cannot be silently converted to benign")
        df["y"] = normalized.isin(pos).astype(int)
    return df


def make_model(name, seed, params=None):
    # tuned params must OVERRIDE the defaults, not collide with them: passing e.g. a tuned
    # n_estimators alongside a hard-coded one raises TypeError, which the HPO fitness then
    # silently swallows as "invalid config" — so merge dicts instead of stacking kwargs.
    params = params or {}
    if name == "LogReg":
        kw = {"max_iter": 1000, "class_weight": "balanced"} | params
        return LogisticRegression(**kw)
    if name == "RandomForest":
        kw = {"n_estimators": 300, "class_weight": "balanced",
              "n_jobs": -1, "random_state": seed} | params
        return RandomForestClassifier(**kw)
    if name == "HistGB":
        kw = {"class_weight": "balanced", "random_state": seed} | params
        return HistGradientBoostingClassifier(**kw)
    if name == "MLP":  # scaled inputs matter for a neural net on tabular features
        pipe = make_pipeline(StandardScaler(),
                             MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=400,
                                           random_state=seed))
        return pipe.set_params(**params) if params else pipe  # keys use mlpclassifier__ prefix
    raise ValueError(f"unknown model: {name}")

def make_any_model(name: str, seed: int, params: dict | None = None):
    p = params or {}
    if name == "Stack[CB+LR]":
        from catboost import CatBoostClassifier
        from sklearn.ensemble import StackingClassifier
        from sklearn.linear_model import LogisticRegression
        return StackingClassifier(
            estimators=[("CB", CatBoostClassifier(iterations=300, depth=6, learning_rate=0.1,
                                                  random_seed=seed, verbose=False)),
                        ("LR", LogisticRegression(max_iter=1000, class_weight="balanced"))],
            final_estimator=LogisticRegression(max_iter=1000),
            stack_method="predict_proba", cv=5, n_jobs=-1)
    if name == "XGBoost":
        from xgboost import XGBClassifier
        return XGBClassifier(**{"n_estimators": 300, "max_depth": 6, "learning_rate": 0.1,
                                "eval_metric": "logloss", "random_state": seed, "n_jobs": -1,
                                **p})
    if name == "LightGBM":
        from lightgbm import LGBMClassifier
        return LGBMClassifier(**{"n_estimators": 300, "num_leaves": 63, "learning_rate": 0.1,
                                 "random_state": seed, "n_jobs": -1, "verbosity": -1, **p})
    if name == "CatBoost":
        from catboost import CatBoostClassifier
        return CatBoostClassifier(**{"iterations": 300, "depth": 6, "learning_rate": 0.1,
                                     "random_seed": seed, "verbose": False, **p})
    return make_model(name, seed, params)

URL_CSV = "data/processed/dataset_url.csv"


ALIGNED = "data/processed/vn_compphish.csv"


def parse_date(s):
    s = (s if isinstance(s, str) else "").strip()[:10]
    for fmt in ("%d/%m/%Y", "%Y-%m-%d"):
        try:
            return datetime.strptime(s, fmt)
        except ValueError:
            pass
    return None


def load():
    meta = pd.read_csv(URL_CSV, usecols=["url_norm", "label", "collected_at"], dtype=str)
    meta["date"] = meta["collected_at"].map(parse_date)
    df = add_label(pd.read_csv(ALIGNED))
    df = df.merge(meta.rename(columns={"url_norm": "url"})[["url", "date"]],
                  on="url", how="left")
    df = df.dropna(subset=[c for c in COMPPHISH if c in df.columns])
    df["rdom"] = df["dom"].astype(str).str.lower().map(registered_domain)
    return df


def split_phishing(ph: pd.DataFrame, cut: float, test_after: str | None = None):
    if test_after:
        d = pd.Timestamp(test_after)
        ph_tr, ph_te = ph[ph.date <= d], ph[ph.date > d]
    else:
        k = int(len(ph) * cut)
        ph_tr, ph_te = ph.iloc[:k], ph.iloc[k:]
    leaked = ph_te.rdom.isin(set(ph_tr.rdom))
    return ph_tr, ph_te[~leaked], int(leaked.sum())

def wilson(k: float, n: int, z: float = 1.96) -> tuple[float, float]:
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    denom = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denom
    return (max(0.0, centre - half), min(1.0, centre + half))

df_check = load()
print(f"{len(df_check):,} rows, {int(df_check.y.sum()):,} phishing, "
      f"{int(df_check[df_check.y == 1].date.notna().sum()):,} dated phishing")

## Bước 2: SHAP theo giao thức đánh giá

Câu hỏi mở đầu của bài: mô hình dựa vào đặc trưng nào, và câu trả lời có đổi theo cách chia train/test không? So hai giao thức trên cùng tập dòng:
- `random_same_rows`: chia ngẫu nhiên tin phishing (cùng tỷ lệ 70/30);
- `temporal_strict`: chia tin phishing theo thời gian, bỏ tên miền trùng khỏi tập kiểm.

Tin benign gần như không có ngày, nên ở cả hai giao thức chúng được chia ngẫu nhiên 70/30. Mỗi (họ mô hình, giao thức, seed) có bộ số ngẫu nhiên riêng,
để hai giao thức không dùng chung một mặt nạ benign. Với mỗi lần chạy, SHAP được tính trên 2.000 dòng kiểm lấy ngẫu nhiên, rồi lấy trung bình |SHAP|
cho từng đặc trưng.

Kết quả gồm: bảng so sánh thứ hạng đặc trưng giữa hai giao thức (CatBoost, trung bình 5 seed), và hệ số Spearman giữa các bảng xếp hạng, so giữa hai
giao thức với giữa các seed của cùng một giao thức. Nếu độ tương quan giữa hai giao thức không thấp hơn độ tương quan giữa các seed, thì khác biệt
giữa giao thức không lớn hơn nhiễu do seed.

In [ ]:
PS_OUT = "data/processed/p6/p6_protocol_shap.csv"


PS_OUT_RUNS = "data/processed/p6/p6_protocol_shap_runs.csv"


PS_OUT_RHO = "data/processed/p6/p6_protocol_shap_rho.csv"


def shap_importance(model, X, feats, sample, seed):
    import shap
    rng = np.random.RandomState(seed)
    sub = X[rng.choice(len(X), size=min(sample, len(X)), replace=False)]
    sv = shap.TreeExplainer(model).shap_values(sub)
    if isinstance(sv, list):  # binary API variants
        sv = sv[1]
    return pd.Series(np.abs(sv).mean(axis=0), index=feats)


def run_protocol_shap(families=['CatBoost', 'XGBoost'], seeds=5, sample=2000, out='data/processed/p6/p6_protocol_shap.csv'):
    args = SimpleNamespace(families=families, seeds=seeds, sample=sample, out=out)
    df = load()
    feats = [c for c in COMPPHISH if c in df.columns]
    ph = df[(df.y == 1) & df.date.notna()].sort_values("date").reset_index(drop=True)
    be = df[df.y == 0].reset_index(drop=True)
    cut = int(len(ph) * 0.70)
    ph_tr, ph_te = ph.iloc[:cut], ph.iloc[cut:]
    ph_te = ph_te[~ph_te.rdom.isin(set(ph_tr.rdom))]
    pool = pd.concat([ph_tr, ph_te])
    def make_split(proto, rng):
        """Benign mask (and, for the control, the phishing mask) drawn from THIS run's rng —
        no sharing across protocols."""
        bmask = rng.rand(len(be)) < 0.70
        if proto == "temporal_strict":
            return pd.concat([ph_tr, be[bmask]]), pd.concat([ph_te, be[~bmask]])
        pmask = rng.rand(len(pool)) < (len(ph_tr) / len(pool))
        return (pd.concat([pool[pmask], be[bmask]]),
                pd.concat([pool[~pmask], be[~bmask]]))
    runs = []   # (family, proto, seed) -> Series
    for fi, fam in enumerate(args.families):
        for pi, proto in enumerate(("random_same_rows", "temporal_strict")):
            for s in range(args.seeds):
                rng = np.random.RandomState(100_000 * (fi + 1) + 10_000 * (pi + 1) + s)
                tr, te = make_split(proto, rng)
                m = make_any_model(fam, s)
                m.fit(tr[feats].to_numpy(float), tr["y"].to_numpy(int))
                imp = shap_importance(m, te[feats].to_numpy(float), feats, args.sample, s)
                runs.append({"family": fam, "protocol": proto, "seed": s, "imp": imp})
                print(f"[i] {fam:<9} {proto:<17} seed={s} trained on {len(tr)}, "
                      f"SHAP over {min(args.sample, len(te))} test rows", flush=True)
    # long-format dump
    long = pd.concat(
        [r["imp"].rename("shap").rename_axis("feature").reset_index()
           .assign(family=r["family"], protocol=r["protocol"], seed=r["seed"])
         for r in runs], ignore_index=True)
    long.to_csv(PS_OUT_RUNS, index=False)
    # per-feature contrast table for the FIRST family, seed-averaged
    fam0 = args.families[0]
    def mean_imp(fam, proto):
        sel = [r["imp"] for r in runs if r["family"] == fam and r["protocol"] == proto]
        return pd.concat(sel, axis=1).mean(axis=1)
    tab = pd.DataFrame({
        "shap_random": mean_imp(fam0, "random_same_rows"),
        "shap_temporal": mean_imp(fam0, "temporal_strict"),
    })
    tab["rank_random"] = tab.shap_random.rank(ascending=False).astype(int)
    tab["rank_temporal"] = tab.shap_temporal.rank(ascending=False).astype(int)
    tab["rank_delta"] = tab.rank_random - tab.rank_temporal
    tab = tab.sort_values("shap_temporal", ascending=False).round(5)
    tab.to_csv(args.out, index_label="feature")
    # rho summary: between-protocol (same-seed pairs) vs within-protocol (re-seed pairs)
    from scipy.stats import spearmanr
    def rho(a, b):
        return spearmanr(a, b).statistic
    rows = []
    for fam in args.families:
        get = lambda proto, s: next(r["imp"] for r in runs
                                    if r["family"] == fam and r["protocol"] == proto
                                    and r["seed"] == s)
        between = [rho(get("random_same_rows", s), get("temporal_strict", s))
                   for s in range(args.seeds)]
        within = {proto: [rho(get(proto, i), get(proto, j))
                          for i, j in itertools.combinations(range(args.seeds), 2)]
                  for proto in ("random_same_rows", "temporal_strict")}
        rows.append({
            "family": fam,
            "rho_between_mean": np.mean(between), "rho_between_sd": np.std(between, ddof=1),
            "rho_within_random_mean": np.mean(within["random_same_rows"]),
            "rho_within_random_sd": np.std(within["random_same_rows"], ddof=1),
            "rho_within_temporal_mean": np.mean(within["temporal_strict"]),
            "rho_within_temporal_sd": np.std(within["temporal_strict"], ddof=1),
            "n_seeds": args.seeds,
        })
        print(f"\n{fam}: rho between-protocol {np.mean(between):.3f}±{np.std(between, ddof=1):.3f}"
              f" | within-random {np.mean(within['random_same_rows']):.3f}"
              f" | within-temporal {np.mean(within['temporal_strict']):.3f}")
    pd.DataFrame(rows).round(4).to_csv(PS_OUT_RHO, index=False)
    print(f"\n{tab.to_string()}")
    print(f"[+] -> {args.out}, {PS_OUT_RUNS}, {PS_OUT_RHO}")

In [ ]:
run_protocol_shap()
for f in ("p6_protocol_shap.csv", "p6_protocol_shap_runs.csv", "p6_protocol_shap_rho.csv"):
    check_file("2 protocol SHAP", f"data/processed/p6/{f}")
display(pd.read_csv("data/processed/p6/p6_protocol_shap_rho.csv"))
report("2 protocol SHAP")

## Bước 3: `tld_len` đọc gì ở đuôi `.vn`

Bước 2 cho thấy `tld_len` chiếm phần lớn sức nặng ở cả hai giao thức. Bước này hỏi nó mã hóa điều gì. Trên phép chia theo thời gian (CatBoost, seed 0):
- tính SHAP của `tld_len` cho mọi dòng kiểm, rồi gom theo đuôi tên miền (gộp `.vn` và các đuôi `.xxx.vn` thành một nhóm), kèm tỷ lệ bỏ sót phishing
  và báo nhầm benign ở ngưỡng 0,5 cho từng nhóm. SHAP âm nghĩa là đuôi đó đẩy dự đoán về phía benign;
- đếm token thương hiệu (tên ngân hàng, ví điện tử...) trong mọi tên miền đăng ký của tin phishing có ngày, kể cả khi token bị chèn gạch ngang hay chữ
  số (`vietcom-bank`). Chỉ dùng token từ 5 ký tự trở lên để tránh trùng ngẫu nhiên.

`split_fit` là phép chia và mô hình dùng chung cho Bước 5, để hai bước giải thích đúng cùng một tập tin bị bỏ sót.

In [ ]:
TLD_OUT = "data/processed/p6/p6_tld_shap.csv"


BRAND_OUT = "data/processed/p6/p6_brand_hits.csv"


BRANDS = "data/processed/brand_tokens.json"


def strip_diacritics(s: str) -> str:
    s = unicodedata.normalize("NFD", s)
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return s.replace("đ", "d").replace("Đ", "D").lower()


def split_fit(df, feats, family: str, seed: int):
    ph = df[(df.y == 1) & df.date.notna()].sort_values("date").reset_index(drop=True)
    be = df[df.y == 0].reset_index(drop=True)
    cut = int(len(ph) * 0.70)
    ph_tr, ph_te = ph.iloc[:cut], ph.iloc[cut:]
    ph_te = ph_te[~ph_te.rdom.isin(set(ph_tr.rdom))]
    rng = np.random.RandomState(seed)
    bmask = rng.rand(len(be)) < 0.70
    tr = pd.concat([ph_tr, be[bmask]])
    te = pd.concat([ph_te, be[~bmask]]).reset_index(drop=True)

    m = make_any_model(family, seed)
    m.fit(tr[feats].to_numpy(float), tr["y"].to_numpy(int))
    return m, tr, te, ph


def run_vn_reading(family='CatBoost', seed=0):
    args = SimpleNamespace(family=family, seed=seed)
    df = load()
    feats = [c for c in COMPPHISH if c in df.columns]
    m, tr, te, ph = split_fit(df, feats, args.family, args.seed)
    # ---- (a) per-TLD signed SHAP of tld_len on the full temporal test set
    import shap
    sv = shap.TreeExplainer(m).shap_values(te[feats].to_numpy(float))
    if isinstance(sv, list):
        sv = sv[1]
    i_tld = feats.index("tld_len")
    te["shap_tld_len"] = sv[:, i_tld]
    te["pred"] = m.predict_proba(te[feats].to_numpy(float))[:, 1]
    te["tld_str"] = te["tld"].astype(str).str.lower()
    # group the .vn second-level family with bare .vn: the registry prior is the same
    te["tld_grp"] = np.where(te.tld_str.str.endswith("vn"), ".vn(+2LD)", "." + te.tld_str)
    rows = []
    for tld, g in te.groupby("tld_grp"):
        gp = g[g.y == 1]
        gb = g[g.y == 0]
        rows.append({
            "tld": tld, "n": len(g), "n_phish": len(gp),
            "phish_rate": round(len(gp) / len(g), 4),
            "mean_shap_tld_len": round(float(g.shap_tld_len.mean()), 5),
            "fnr_phish@0.5": round(float((gp.pred < 0.5).mean()), 4) if len(gp) else np.nan,
            # the benign-side price of the same operating point, per group, so the table can
            # show both error rates rather than the miss rate alone
            "fpr_benign@0.5": round(float((gb.pred >= 0.5).mean()), 4) if len(gb) else np.nan,
        })
    tab = pd.DataFrame(rows).sort_values("n", ascending=False)
    tab.to_csv(TLD_OUT, index=False)
    print("=== per-TLD (phishing-temporal test set) ===")
    print(tab.head(15).to_string(index=False))
    vn = te[(te.y == 1) & te.tld_str.str.endswith("vn")]
    other = te[(te.y == 1) & ~te.tld_str.str.endswith("vn")]
    print(f"\n.vn phishing in test: {len(vn)}  FNR@0.5={float((vn.pred < 0.5).mean()):.3f}   "
          f"other-TLD phishing: {len(other)}  FNR@0.5={float((other.pred < 0.5).mean()):.3f}")
    # ---- (c) brand tokens over ALL dated phishing registered domains (train+test: descriptive)
    toks = json.load(open(BRANDS))["tokens"]
    # tokens are already ascii-ish; keep len>=5 to avoid junk substring hits (e.g. 'vien')
    toklist = sorted({t["token"].lower() for t in toks if len(t["token"]) >= 5})
    doms = ph["rdom"].astype(str).str.lower().unique()
    hits = []
    for d in doms:
        flat = strip_diacritics(d)
        squash = re.sub(r"[-\d.]", "", flat)
        for t in toklist:
            if t in flat:
                hits.append({"brand": t, "domain": d, "mangled": 0})
            elif t in squash:  # token only appears once dashes/digits are removed
                hits.append({"brand": t, "domain": d, "mangled": 1})
    hb = pd.DataFrame(hits)
    hb.to_csv(BRAND_OUT, index=False)
    top = (hb.groupby("brand").agg(domains=("domain", "nunique"), mangled=("mangled", "sum"))
             .sort_values("domains", ascending=False))
    print(f"\n=== brand tokens in {len(doms)} dated phishing registrable domains ===")
    print(f"domains with >=1 brand hit: {hb.domain.nunique()} "
          f"({hb.domain.nunique() / len(doms):.1%});  mangled hits: {int(hb.mangled.sum())}")
    print(top.head(15).to_string())
    print(f"[+] -> {TLD_OUT}, {BRAND_OUT}")

In [ ]:
run_vn_reading()
for f in ("p6_tld_shap.csv", "p6_brand_hits.csv"):
    check_file("3 vn reading", f"data/processed/p6/{f}")
report("3 vn reading")

## Bước 4: Ngưỡng quyết định theo nhóm

Nếu mô hình bỏ sót nhiều tin `.vn`, có sửa được bằng cách đặt ngưỡng riêng cho nhóm `.vn` không? Phép chia: 70% tin phishing cũ nhất để huấn luyện, trong
đó 15% mới nhất được giữ lại làm tập hiệu chỉnh (calibration); tin benign chia ngẫu nhiên theo cùng tỷ lệ. Mọi quy tắc được so ở cùng một mức báo nhầm:
`alpha` là tỷ lệ benign của tập hiệu chỉnh bị báo nhầm khi dùng ngưỡng 0,5.

Các quy tắc:
- `default`: ngưỡng 0,5 cho mọi tin;
- `global`: một ngưỡng chung đặt tại phân vị `1 - alpha` của điểm benign hiệu chỉnh;
- `per-group`: mỗi nhóm (`.vn`, phần còn lại) một ngưỡng tại phân vị `1 - alpha` của điểm benign của chính nhóm đó;
- `vn-only`: cho nhóm `.vn` cùng mức báo nhầm mà nhóm còn lại đang có ở ngưỡng 0,5, giữ nguyên nhóm còn lại;
- `mapped`: đưa phân phối điểm benign của nhóm `.vn` về phân phối của nhóm còn lại (ánh xạ theo phân vị), rồi dùng một ngưỡng chung;
- `balanced`: huấn luyện lại với trọng số cân bằng tám ô (bốn tầng đuôi × hai lớp), rồi đặt ngưỡng ở cùng mức báo nhầm.

Bước này chạy hai lần: với CatBoost (bảng chính của bài) và với mô hình xếp chồng CatBoost + LogReg (kiểm xem điểm mù có phụ thuộc họ mô hình không).
Bài chỉ dùng kết quả CatBoost của bước này, nên chỉ file đó được so. Lần chạy CatBoost + LogReg vẫn được giữ để người đọc xem; bảng kết quả cũ của lần chạy
đó được tạo trước khi trọng số cân bằng đổi từ 4 ô sang 8 ô, nên không so với nó.

In [ ]:
GT_OUT_CSV = os.path.join(ROOT, "data", "processed", "p6", "p6_group_threshold.csv")


STRATA = ("vn_short", "cc_short", "vn_compound", "long")


def vn_group(tld: pd.Series) -> np.ndarray:
    return tld.astype(str).str.lower().str.endswith("vn").to_numpy()


def strata_of(frame: pd.DataFrame) -> pd.Series:
    sl = frame["tld"].astype(str).str.lower().str.lstrip(".").str.len().to_numpy()
    vn = vn_group(frame["tld"])
    out = np.where(sl == 2,
                   np.where(vn, "vn_short", "cc_short"),
                   np.where(vn, "vn_compound", "long"))
    return pd.Series(out, index=frame.index)


def stratum_class_weights(frame: pd.DataFrame) -> np.ndarray:
    groups = strata_of(frame).to_numpy()
    labels = frame["y"].to_numpy(int)
    weights = np.ones(len(frame), dtype=float)
    for group in STRATA:
        for label in (0, 1):
            cell = (groups == group) & (labels == label)
            if cell.sum():
                weights[cell] = len(frame) / (len(STRATA) * 2.0 * cell.sum())
    return weights


def split_and_fit(df, feats, family: str, seed: int, cal_frac: float):
    ph = df[(df.y == 1) & df.date.notna()].sort_values("date").reset_index(drop=True)
    be = df[df.y == 0].reset_index(drop=True)
    cut = int(len(ph) * 0.70)
    ph_tr_all, ph_te = ph.iloc[:cut], ph.iloc[cut:]
    ph_te = ph_te[~ph_te.rdom.isin(set(ph_tr_all.rdom))]
    ccut = int(len(ph_tr_all) * (1 - cal_frac))
    ph_fit, ph_cal = ph_tr_all.iloc[:ccut], ph_tr_all.iloc[ccut:]  # calibration = newest train

    rng = np.random.RandomState(seed)
    r = rng.rand(len(be))
    be_fit = be[r < 0.70 * (1 - cal_frac)]
    be_cal = be[(r >= 0.70 * (1 - cal_frac)) & (r < 0.70)]
    be_te = be[r >= 0.70]

    tr = pd.concat([ph_fit, be_fit])
    m = make_any_model(family, seed)
    m.fit(tr[feats].to_numpy(float), tr["y"].to_numpy(int))

    cal = pd.concat([ph_cal, be_cal]).reset_index(drop=True)
    te = pd.concat([ph_te, be_te]).reset_index(drop=True)
    cal["pred"] = m.predict_proba(cal[feats].to_numpy(float))[:, 1]
    te["pred"] = m.predict_proba(te[feats].to_numpy(float))[:, 1]
    cal["vn"] = vn_group(cal["tld"])
    te["vn"] = vn_group(te["tld"])
    return tr, m, cal, te


def threshold_rules(cal_be: pd.DataFrame) -> tuple[dict, float]:
    alpha = float((cal_be.pred >= 0.5).mean())  # the 0.5 baseline's own false-alarm budget
    q = 1 - alpha

    def bq(scores: pd.Series) -> float:
        return float(scores.quantile(q)) if len(scores) else 0.5

    # "vn-only": give .vn pages the same operating point non-.vn pages already get at 0.5
    # (threshold at the quantile matching the OTHER group's realized calibration FPR), leave
    # everything else untouched — total FPR is allowed to rise; that rise is the price tag.
    fpr_other_cal = float((cal_be[~cal_be.vn].pred >= 0.5).mean())
    return {
        "default": {"vn": 0.5, "other": 0.5},
        "global": {"vn": bq(cal_be.pred), "other": bq(cal_be.pred)},
        "per-group": {"vn": bq(cal_be[cal_be.vn].pred), "other": bq(cal_be[~cal_be.vn].pred)},
        "vn-only": {"vn": float(cal_be[cal_be.vn].pred.quantile(1 - fpr_other_cal))
                    if len(cal_be[cal_be.vn]) else 0.5, "other": 0.5},
    }, alpha


def gt_one_seed(df, feats, family: str, seed: int, cal_frac: float,
             return_scored: bool = False):
    tr, m, cal, te = split_and_fit(df, feats, family, seed, cal_frac)

    # STRATUM-BALANCED TRAINING — the only non-post-hoc remedy here. The old arm reweighted four
    # (.vn/non-.vn, class) cells, exactly the pooled grouping Section V-C shows to be invalid.
    # Reweight all eight (suffix stratum, class) cells instead, then threshold at the same matched
    # budget as `global` so the contrast isolates TRAINING from thresholding.
    # PREDICTION, recorded before the run: should NOT beat per-group thresholding at the operating
    # point (\S ssec:groupthr shows the residual is a budget constraint). Open question is the ROC
    # itself (baseline auc_vn 0.855): if it rises, a training-time fix exists; if not, the
    # constraint is intrinsic to the 21-feature space, strengthening the infrastructure-cascade case.
    ytr = tr["y"].to_numpy(int)
    w = stratum_class_weights(tr)
    mb = make_any_model(family, seed)
    mb.fit(tr[feats].to_numpy(float), ytr, sample_weight=w)

    cal["pred_bal"] = mb.predict_proba(cal[feats].to_numpy(float))[:, 1]
    te["pred_bal"] = mb.predict_proba(te[feats].to_numpy(float))[:, 1]

    cal_be = cal[cal.y == 0]
    thr, alpha = threshold_rules(cal_be)
    q = 1 - alpha

    def bq(scores: pd.Series) -> float:
        return float(scores.quantile(q)) if len(scores) else 0.5

    # GROUP-CONDITIONAL BENIGN QUANTILE MAPPING, not per-group calibration: calibration needs
    # positives per group and the .vn calibration slice holds 11 phishing vs 1,033 benign —
    # isotonic/Platt on 11 positives is overfitting. So map each group's BENIGN score
    # distribution onto a common reference, generalising the per-group threshold from one
    # operating point to the whole benign-side curve (what a cascade gate needs to compose).
    # PREDICTION, stated before the number is read: at the matched budget this lands on top of
    # the per-group rule (both place each group at the same benign quantile); it buys
    # composability, not a better operating point.
    def qmap(cal_scores: pd.Series, ref_scores: pd.Series, x: np.ndarray) -> np.ndarray:
        """Map x through the benign quantile function of its own group onto the reference."""
        if not len(cal_scores) or not len(ref_scores):
            return x
        u = np.searchsorted(np.sort(cal_scores.to_numpy()), x, side="right") / len(cal_scores)
        return np.quantile(ref_scores.to_numpy(), np.clip(u, 0.0, 1.0))

    ref_be = cal_be[~cal_be.vn]["pred"]              # non-.vn benign is the reference frame
    te_mapped = np.where(
        te.vn.to_numpy(),
        qmap(cal_be[cal_be.vn]["pred"], ref_be, te.pred.to_numpy()),
        te.pred.to_numpy())

    fpr_other_cal = float((cal_be[~cal_be.vn].pred >= 0.5).mean())
    # The mapped rule scores on te_mapped with ONE threshold for both groups (the property under
    # test). The threshold must come from calibration benign mapped the SAME way, per group: a
    # first version pushed pooled benign through the .vn quantile function alone, putting the
    # threshold far too high (FPR 0.024 against a 0.26 budget).
    cal_be_mapped = np.where(
        cal_be.vn.to_numpy(),
        qmap(cal_be[cal_be.vn]["pred"], ref_be, cal_be["pred"].to_numpy()),
        cal_be["pred"].to_numpy())
    thr_mapped = bq(pd.Series(cal_be_mapped))
    # The balanced model needs its own budget-matched threshold: its scores live on a different
    # scale, so reusing the baseline's would confound training with thresholding.
    thr_bal = float(cal_be["pred_bal"].quantile(1 - alpha)) if len(cal_be) else 0.5

    # Per-group ROC area, carried on every row so the paper's "the .vn group ranks BETTER than
    # the group the model is not blind to" is a checkable number and not a reading of a figure.
    # Threshold-independent by construction, hence identical across the four conditions.
    from sklearn.metrics import roc_auc_score
    auc = {}
    for gkey, gmask in (("vn", te.vn.to_numpy()), ("other", ~te.vn.to_numpy())):
        sub = te[gmask]
        for col, suffix in (("pred", ""), ("pred_bal", "_bal")):
            auc[gkey + suffix] = (
                round(float(roc_auc_score(sub.y.to_numpy(int), sub[col].to_numpy(float))), 4)
                if sub.y.nunique() > 1 else float("nan"))
    te_strata = strata_of(te)
    for skey in STRATA:
        sub = te[(te_strata == skey).to_numpy()]
        for col, suffix in (("pred", ""), ("pred_bal", "_bal")):
            auc[skey + suffix] = (
                round(float(roc_auc_score(sub.y.to_numpy(int), sub[col].to_numpy(float))), 4)
                if sub.y.nunique() > 1 else float("nan"))

    rows = []
    for cond, t in list(thr.items()) + [("mapped", None), ("balanced", None)]:
        if cond == "mapped":
            scores, thr_vec = te_mapped, np.full(len(te), thr_mapped)
            t = {"vn": thr_mapped, "other": thr_mapped}
        elif cond == "balanced":
            scores = te["pred_bal"].to_numpy()
            thr_vec = np.full(len(te), thr_bal)
            t = {"vn": thr_bal, "other": thr_bal}
        else:
            scores, thr_vec = te.pred.to_numpy(), np.where(te.vn, t["vn"], t["other"])
        pred = (scores >= thr_vec).astype(int)
        y = te.y.to_numpy(int)
        vn_ph = te.vn.to_numpy() & (y == 1)
        ot_ph = ~te.vn.to_numpy() & (y == 1)
        vn_be = te.vn.to_numpy() & (y == 0)
        ot_be = ~te.vn.to_numpy() & (y == 0)
        row = {
            "seed": seed, "condition": cond, "alpha_cal": round(alpha, 4),
            "balance_cells": "suffix_stratum_x_class",
            "auc_vn": auc["vn"], "auc_other": auc["other"],
            "auc_vn_bal": auc["vn_bal"], "auc_other_bal": auc["other_bal"],
            "rec_vn": round(float((pred[te.vn.to_numpy() & (te.y.to_numpy() == 1)] == 1).mean()),
                            4),
            "rec_other": round(
                float((pred[~te.vn.to_numpy() & (te.y.to_numpy() == 1)] == 1).mean()), 4),
            "fpr_other_cal@0.5": round(fpr_other_cal, 4),
            "thr_vn": round(t["vn"], 4), "thr_other": round(t["other"], 4),
            "n_vn_phish": int(vn_ph.sum()),
            "fnr_vn": round(float((pred[vn_ph] == 0).mean()), 4),
            "fnr_other": round(float((pred[ot_ph] == 0).mean()), 4),
            "fpr": round(float((pred[y == 0] == 1).mean()), 4),
            "fpr_vn": round(float((pred[vn_be] == 1).mean()), 4),
            "fpr_other": round(float((pred[ot_be] == 1).mean()), 4),
            "f1": round(_f1(y, pred), 4),
        }
        for skey in STRATA:
            sm = (te_strata == skey).to_numpy()
            sph, sbe = sm & (y == 1), sm & (y == 0)
            row[f"auc_{skey}"] = auc[skey]
            row[f"auc_{skey}_bal"] = auc[skey + "_bal"]
            row[f"fnr_{skey}"] = (round(float((pred[sph] == 0).mean()), 4)
                                    if sph.sum() else float("nan"))
            row[f"fpr_{skey}"] = (round(float((pred[sbe] == 1).mean()), 4)
                                    if sbe.sum() else float("nan"))
        rows.append(row)
    # The ROC figure needs the scores the table was computed from, not a re-derivation: a second
    # split would be a second experiment, and the operating points it marked would not be the
    # ones the table reports.
    return (rows, te[["y", "vn", "pred"]].copy(), thr) if return_scored else rows


def _f1(y, pred) -> float:
    from sklearn.metrics import f1_score
    return float(f1_score(y, pred, zero_division=0))


def run_group_threshold(family='CatBoost', seeds=5, cal_frac=0.15):
    args = SimpleNamespace(family=family, seeds=seeds, cal_frac=cal_frac)
    df = load()
    feats = [c for c in COMPPHISH if c in df.columns]
    rows = []
    scored = []
    for s in range(args.seeds):
        got, te_s, thr_s = gt_one_seed(df, feats, args.family, s, args.cal_frac,
                                    return_scored=True)
        rows += got
        scored.append((te_s, thr_s))
        r = rows[-1]
        print(f"[i] seed {s}: alpha={r['alpha_cal']:.3f} thr_vn={rows[-1]['thr_vn']:.3f} | "
              + " ".join(f"{x['condition']}: fnr_vn={x['fnr_vn']:.3f} fpr={x['fpr']:.3f}"
                         for x in rows[-4:]))
    res = pd.DataFrame(rows)
    os.makedirs(os.path.dirname(GT_OUT_CSV), exist_ok=True)
    if args.family != "CatBoost":
        # A non-canonical family (e.g. Stack[CB+LR]) gets its own suffixed CSV
        import re as _re
        slug = _re.sub(r"\W+", "", args.family).lower()
        out_csv = GT_OUT_CSV.replace(".csv", f"_{slug}.csv")
        res.to_csv(out_csv, index=False)
        print(f"[+] {out_csv}")
        return
    res.to_csv(GT_OUT_CSV, index=False)
    print(f"[+] {GT_OUT_CSV}")

In [ ]:
run_group_threshold()
run_group_threshold(family="Stack[CB+LR]")
check_file("4 group threshold", "data/processed/p6/p6_group_threshold.csv")
display(pd.read_csv("data/processed/p6/p6_group_threshold.csv").groupby("condition")[["fnr_vn", "fnr_other", "fpr", "f1"]].mean().round(3))
report("4 group threshold")

## Bước 5: Thâm hụt của từng tin `.vn` bị bỏ sót

Với mỗi tin phishing `.vn` bị bỏ sót ở ngưỡng 0,5 (cùng mô hình và phép chia với Bước 3), câu hỏi là: riêng đuôi tên miền có đủ giải thích vì sao tin đó
bị bỏ sót không? Phép so phải làm trong không gian log-odds, vì chỉ ở đó tổng các giá trị SHAP cộng lại đúng bằng đầu ra của mô hình (notebook kiểm điều
này cho từng dòng, sai số dưới 1e-6). So trong không gian xác suất sẽ cho ra những con số trông hợp lý nhưng sai về bản chất.

Hai cách đọc, cho kết quả ở hai phía:
- kế toán cộng: `-SHAP(tld_len)` có lớn hơn hoặc bằng khoảng cách từ dòng đó tới ngưỡng không (cách đọc bảo thủ);
- phản thực tế: đặt `tld_len` của dòng đó bằng giá trị phổ biến nhất của các đuôi không phải `.vn` trong tập huấn luyện, giữ nguyên mọi thứ khác, chấm lại
  và xem tin có vượt ngưỡng không (cách đọc rộng rãi).

Thêm một phép quét: đặt `tld_len` lần lượt bằng mọi giá trị có trong corpus, tách riêng `.vn` trần và họ `.com.vn`. Ngoài ra, bước này đọc ngưỡng `.vn`
đã hiệu chỉnh từ Bước 4 để báo thêm một điểm vận hành, nên phải chạy sau Bước 4.

In [ ]:
PROC = os.path.join(ROOT, "data", "processed")


GROUPTHR_CSV = os.path.join(PROC, "p6", "p6_group_threshold.csv")


FEATURE = "tld_len"          # the dominant feature of Table~\ref{tab:shapcontrast}


def logit(p: float) -> float:
    return math.log(p / (1.0 - p))


def margin_of(model, X: np.ndarray) -> np.ndarray:
    try:
        return np.asarray(model.predict(X, prediction_type="RawFormulaVal"), dtype=float)
    except TypeError:
        return np.asarray(model.decision_function(X), dtype=float)


def neutralised_margin(model, X: np.ndarray, j: int, value: float) -> np.ndarray:
    Xc = X.copy()
    Xc[:, j] = value
    return margin_of(model, Xc)


def run_vn_deficit(family='CatBoost', seed=0, thr=0.5):
    args = SimpleNamespace(family=family, seed=seed, thr=thr)
    df = load()
    feats = [c for c in COMPPHISH if c in df.columns]
    if FEATURE not in feats:
        raise SystemExit(f"feature {FEATURE!r} not in the model's feature list: {feats}")
    j = feats.index(FEATURE)
    m, tr, te, _ph = split_fit(df, feats, args.family, args.seed)
    X_te = te[feats].to_numpy(float)
    te["margin"] = margin_of(m, X_te)
    te["pred"] = m.predict_proba(X_te)[:, 1]
    te["vn"] = te["tld"].astype(str).str.lower().str.endswith("vn")
    # sanity: the raw output must BE the logit of the reported probability, or the whole
    # margin-space argument is being made against the wrong quantity
    assert np.allclose(1.0 / (1.0 + np.exp(-te["margin"].to_numpy())),
                       te["pred"].to_numpy(), atol=1e-8), "raw output is not logit(p)"
    vn_ph = te[te.vn & (te.y == 1)].copy()
    fn = vn_ph[vn_ph.pred < args.thr].copy()
    print(f"[i] .vn phishing in test: {len(vn_ph)}   missed at {args.thr}: {len(fn)} "
          f"({len(fn) / len(vn_ph):.3f})")
    # ---- per-row SHAP in margin space, with the additivity identity checked, not assumed
    import shap
    ex = shap.TreeExplainer(m)
    Xf = fn[feats].to_numpy(float)
    sv = ex.shap_values(Xf)
    if isinstance(sv, list):
        sv = sv[1]
    base = float(np.ravel(ex.expected_value)[0])
    recon = sv.sum(axis=1) + base
    err = float(np.max(np.abs(recon - fn["margin"].to_numpy())))
    assert err < 1e-6, f"TreeSHAP additivity violated in margin space (max err {err:.2e})"
    print(f"[i] additivity check passed (max |sum(phi)+phi0 - margin| = {err:.2e})")
    fn["phi_tld_len"] = sv[:, j]
    fn["phi_others"] = sv.sum(axis=1) - sv[:, j]
    fn["deficit"] = logit(args.thr) - fn["margin"]     # > 0 by construction on a miss
    # ---- the neutralisation counterfactual
    tr_ot = tr[~tr["tld"].astype(str).str.lower().str.endswith("vn")]
    neutral = float(tr_ot[FEATURE].mode().iloc[0])     # the commodity suffix the fit window sees
    fn["margin_neutral"] = neutralised_margin(m, Xf, j, neutral)
    fn["delta_margin"] = fn["margin_neutral"] - fn["margin"]
    fn["crossed"] = (fn["margin_neutral"] >= logit(args.thr)).astype(int)
    # the accounting statements
    fn["covered_signed"] = (-fn["phi_tld_len"] >= fn["deficit"]).astype(int)
    fn["covered_abs"] = (fn["phi_tld_len"].abs() >= fn["deficit"]).astype(int)
    n = len(fn)
    k_sig = int(fn.covered_signed.sum())
    k_abs = int(fn.covered_abs.sum())
    k_cr = int(fn.crossed.sum())
    n_benign_ward = int((fn.phi_tld_len < 0).sum())
    lo_s, hi_s = wilson(k_sig, n)
    lo_c, hi_c = wilson(k_cr, n)
    rho = float(pd.Series(-fn.phi_tld_len.to_numpy()).corr(
        pd.Series(fn.delta_margin.to_numpy()), method="spearman"))
    # run, read rather than retyped. Every deficit shifts by the constant logit(t), so this is a
    # translation of the same distribution, not a second experiment.
    thr2 = None
    if os.path.exists(GROUPTHR_CSV):
        g = pd.read_csv(GROUPTHR_CSV)
        g = g[g.condition == "per-group"]
        if len(g):
            thr2 = float(g.thr_vn.mean())
    sec = None
    if thr2 and 0 < thr2 < 1:
        still = fn[fn.margin < logit(thr2)].copy()
        still["deficit2"] = logit(thr2) - still["margin"]
        cov2 = int((-still.phi_tld_len >= still.deficit2).sum())
        cr2 = int((still.margin_neutral >= logit(thr2)).sum())
        sec = (thr2, len(still), cov2, cr2)
        print(f"[i] at the calibrated .vn threshold {thr2:.3f}: {len(still)} still missed, "
              f"{cov2} accounted, {cr2} cross under neutralisation")
    # ---- does the neutral value carry the result?
    # Split by suffix family, because the .vn(+2LD) group pools TWO feature values (2 for bare
    # .vn, 6 for the .com.vn family, per Section 5.2): a pooled sweep would show 45 crossings at
    # tld_len=2 and read as ``even the .vn value flips a third of them'', when in fact that value
    # is a no-op for the 91 rows that already carry it and the crossings are entirely the
    # compound-suffix rows being moved off their own value.
    bare = (fn[FEATURE].to_numpy(float) <= 2.5)
    sweep = []
    for v in sorted(pd.unique(df[FEATURE].dropna().astype(float))):
        if v > 12:
            continue
        mg = neutralised_margin(m, Xf, j, float(v))
        ok = mg >= logit(args.thr)
        sweep.append({"tld_len": float(v), "n_fn": n,
                      "crossed": int(ok.sum()),
                      "n_bare": int(bare.sum()), "crossed_bare": int(ok[bare].sum()),
                      "n_compound": int((~bare).sum()),
                      "crossed_compound": int(ok[~bare].sum()),
                      "mean_delta_margin": float(np.mean(mg - fn["margin"].to_numpy()))})
    sweep = pd.DataFrame(sweep)
    os.makedirs(PROC, exist_ok=True)
    cols = ["rdom", "tld", "pred", "margin", "deficit", "phi_tld_len", "phi_others",
            "margin_neutral", "delta_margin", "crossed", "covered_signed", "covered_abs"]
    fn[[c for c in cols if c in fn.columns]].to_csv(
        os.path.join(PROC, "p6", "p6_vn_deficit.csv"), index=False)
    sweep.to_csv(os.path.join(PROC, "p6", "p6_vn_deficit_sweep.csv"), index=False)
    print(f"[+] data/processed/p6/p6_vn_deficit.csv ({n} rows)")
    print("[+] data/processed/p6/p6_vn_deficit_sweep.csv")
    print(f"[+] accounted for by phi({FEATURE}) alone: {k_sig}/{n} = {k_sig / n:.3f} "
          f"[{lo_s:.3f}, {hi_s:.3f}]")
    print(f"[+] cross the threshold under neutralisation to tld_len={neutral:g}: "
          f"{k_cr}/{n} = {k_cr / n:.3f} [{lo_c:.3f}, {hi_c:.3f}]")
    print(f"[+] Spearman(-phi, delta margin) = {rho:.3f}; "
          f"benign-ward phi on {n_benign_ward}/{n} rows")

In [ ]:
run_vn_deficit()
for f in ("p6_vn_deficit.csv", "p6_vn_deficit_sweep.csv"):
    check_file("5 vn deficit", f"data/processed/p6/{f}")
report("5 vn deficit")

## Bước 6: Điểm mù theo bốn tầng đuôi tên miền

Gộp mọi đuôi `.vn` thành một nhóm che mất một điều: `.vn` trần và `.com.vn` có `tld_len` khác nhau (2 và 6), nên mô hình đối xử với chúng khác hẳn. Bước
này tách thành bốn tầng (`vn_short`, `cc_short`, `vn_compound`, `long`) trên cùng phép chia và mô hình của Bước 4, rồi báo:
- tỷ lệ bỏ sót, báo nhầm và AUC cho từng tầng;
- AUC của nhóm `.vn` gộp so với từng tầng, kèm tỷ lệ cặp (phishing, benign) nằm ở hai tầng khác nhau. Đó là những cặp mà AUC gộp chấm điểm nhưng AUC từng
  tầng thì không, và là nguồn gốc của nghịch lý Simpson trong bài;
- mọi quy tắc ngưỡng (cả theo độ dài đuôi và theo tầng) cùng hai bộ phân loại tầm thường (đoán tất cả phishing, đoán tất cả benign) làm mốc;
- rò rỉ benign: bao nhiêu tin benign trong tập kiểm có tên miền hoặc nguyên vector đặc trưng trùng với một tin benign trong tập huấn luyện, và AUC khi bỏ chúng đi;
- vị trí token thương hiệu trong URL: tên miền đăng ký, tên miền con, hay đường dẫn;
- SHAP của `tld_len` theo từng tầng, trên mô hình đơn của Bước 3.

Chạy hai lần như Bước 4: CatBoost và CatBoost + LogReg.

In [ ]:
STRATA_OUT = os.path.join(ROOT, "data", "processed", "p6", "p6_suffix_strata.csv")


AUC_OUT = os.path.join(ROOT, "data", "processed", "p6", "p6_suffix_auc.csv")


THR_OUT = os.path.join(ROOT, "data", "processed", "p6", "p6_suffix_threshold.csv")


LEAK_OUT = os.path.join(ROOT, "data", "processed", "p6", "p6_leakage.csv")


LOCUS_OUT = os.path.join(ROOT, "data", "processed", "p6", "p6_brand_locus.csv")


def _auc(y, s) -> float:
    from sklearn.metrics import roc_auc_score
    y = np.asarray(y, int)
    return float(roc_auc_score(y, s)) if len(np.unique(y)) > 1 else float("nan")


def _metrics(y, pred) -> dict:
    from sklearn.metrics import f1_score, matthews_corrcoef, balanced_accuracy_score
    return {
        "f1": round(float(f1_score(y, pred, zero_division=0)), 4),
        "mcc": round(float(matthews_corrcoef(y, pred)), 4) if len(np.unique(pred)) > 1 else 0.0,
        "bal_acc": round(float(balanced_accuracy_score(y, pred)), 4),
    }


def strata_rows(te: pd.DataFrame, seed: int) -> list[dict]:
    y = te.y.to_numpy(int)
    pred = (te.pred.to_numpy(float) >= 0.5)
    g = strata_of(te)
    rows = []
    for name in STRATA:
        m = (g == name).to_numpy()
        ph, be = m & (y == 1), m & (y == 0)
        rows.append({
            "seed": seed, "stratum": name, "n": int(m.sum()),
            "n_phish": int(ph.sum()), "n_benign": int(be.sum()),
            "fnr": round(float(1 - pred[ph].mean()), 4) if ph.sum() else np.nan,
            "fpr": round(float(pred[be].mean()), 4) if be.sum() else np.nan,
            "auc": round(_auc(y[m], te.pred.to_numpy(float)[m]), 4),
        })
    return rows


def auc_row(te: pd.DataFrame, seed: int) -> dict:
    y = te.y.to_numpy(int)
    s = te.pred.to_numpy(float)
    g = strata_of(te).to_numpy()
    vn = vn_group(te["tld"])
    short = (g == "vn_short") | (g == "cc_short")
    pb, nb = int(((g == "vn_short") & (y == 1)).sum()), int(((g == "vn_short") & (y == 0)).sum())
    pc = int(((g == "vn_compound") & (y == 1)).sum())
    nc = int(((g == "vn_compound") & (y == 0)).sum())
    tot = (pb + pc) * (nb + nc)
    return {
        "seed": seed,
        "auc_vn_pooled": round(_auc(y[vn], s[vn]), 4),
        "auc_vn_short": round(_auc(y[g == "vn_short"], s[g == "vn_short"]), 4),
        "auc_vn_compound": round(_auc(y[g == "vn_compound"], s[g == "vn_compound"]), 4),
        "auc_other_pooled": round(_auc(y[~vn], s[~vn]), 4),
        "auc_cc_short": round(_auc(y[g == "cc_short"], s[g == "cc_short"]), 4),
        "auc_long": round(_auc(y[g == "long"], s[g == "long"]), 4),
        "auc_short_all": round(_auc(y[short], s[short]), 4),
        "cross_pair_share": round(float((pb * nc + pc * nb) / tot), 4) if tot else np.nan,
        "n_vn_short_phish": pb, "n_vn_compound_phish": pc,
    }


def threshold_rows(cal: pd.DataFrame, te: pd.DataFrame, seed: int) -> list[dict]:
    cal_be = cal[cal.y == 0]
    published, alpha = threshold_rules(cal_be)
    q = 1 - alpha

    def bq(scores: pd.Series) -> float:
        return float(scores.quantile(q)) if len(scores) else 0.5

    cal_g, te_g = strata_of(cal), strata_of(te)
    cal_short = (cal_g == "vn_short") | (cal_g == "cc_short")
    te_short = ((te_g == "vn_short") | (te_g == "cc_short")).to_numpy()

    y = te.y.to_numpy(int)
    s = te.pred.to_numpy(float)
    vn = vn_group(te["tld"])

    # Rule set. Each entry maps to a per-row threshold vector on the SAME scores, so the only
    # thing that varies is where the operating point is placed.
    rules: dict[str, np.ndarray] = {}
    for name in ("default", "global", "per-group", "vn-only"):
        t = published[name]
        rules[name] = np.where(vn, t["vn"], t["other"])
    # (S) the group the model is actually blind to: two-character suffixes, .vn or not.
    t_short = bq(cal_be[cal_short[cal_be.index]].pred)
    t_longg = bq(cal_be[~cal_short[cal_be.index]].pred)
    rules["per-suffixlen"] = np.where(te_short, t_short, t_longg)
    # ... and the full four-way split, which separates the two .vn strata the paper pooled.
    thr4 = {k: bq(cal_be[(cal_g[cal_be.index] == k)].pred) for k in STRATA}
    rules["per-stratum"] = te_g.map(thr4).to_numpy(float)

    rows = []
    for name, tvec in rules.items():
        pred = (s >= tvec).astype(int)
        r = {"seed": seed, "rule": name, "alpha_cal": round(alpha, 4),
             "thr_vn_short": round(float(np.median(tvec[(te_g == "vn_short").to_numpy()])), 4)
             if (te_g == "vn_short").any() else np.nan,
             "thr_cc_short": round(float(np.median(tvec[(te_g == "cc_short").to_numpy()])), 4)
             if (te_g == "cc_short").any() else np.nan,
             "fpr": round(float(pred[y == 0].mean()), 4)}
        for k in STRATA:
            m = (te_g == k).to_numpy() & (y == 1)
            r["fnr_" + k] = round(float((pred[m] == 0).mean()), 4) if m.sum() else np.nan
        r["fnr_short_all"] = round(float((pred[te_short & (y == 1)] == 0).mean()), 4)
        # the pooled .vn FNR the published table prints, so the two tables are checkably the
        # same experiment read two ways
        r["fnr_vn_pooled"] = round(float((pred[vn & (y == 1)] == 0).mean()), 4)
        r.update(_metrics(y, pred))
        rows.append(r)
    # (B) the floor. all-positive is the one that matters -- this test set is ~55% phishing, so
    # its F1 sits within 0.01 of the published per-group rule's.
    for name, pred in (("all-positive", np.ones(len(y), int)),
                       ("all-negative", np.zeros(len(y), int))):
        r = {"seed": seed, "rule": name, "alpha_cal": round(alpha, 4),
             "thr_vn_short": np.nan, "thr_cc_short": np.nan,
             "fpr": round(float(pred[y == 0].mean()), 4)}
        for k in STRATA:
            m = (te_g == k).to_numpy() & (y == 1)
            r["fnr_" + k] = round(float((pred[m] == 0).mean()), 4) if m.sum() else np.nan
        r["fnr_short_all"] = round(float((pred[te_short & (y == 1)] == 0).mean()), 4)
        r["fnr_vn_pooled"] = round(float((pred[vn & (y == 1)] == 0).mean()), 4)
        r.update(_metrics(y, pred))
        rows.append(r)
    return rows


def leakage_rows(tr: pd.DataFrame, te: pd.DataFrame, feats: list[str], seed: int) -> list[dict]:
    tr_be, te_be = tr[tr.y == 0], te[te.y == 0]
    dom_tr = set(tr_be.rdom.astype(str))
    dup_dom_rows = te_be.rdom.astype(str).isin(dom_tr).to_numpy()
    n_dom_te = int(te_be.rdom.nunique())
    n_dom_shared = int(te_be[dup_dom_rows].rdom.nunique())

    key_tr = set(map(tuple, np.round(tr_be[feats].to_numpy(float), 6)))
    dup_vec_rows = np.array([tuple(r) in key_tr
                             for r in np.round(te_be[feats].to_numpy(float), 6)])

    # Two twin-free variants, because the two exclusions do not mean the same thing. Dropping
    # DOMAIN twins is the conservative, defensible one: those rows the model could genuinely have
    # memorised. Dropping FEATURE-VECTOR twins is the aggressive one -- 21 coarse integers
    # collide by coincidence as well as by memorisation, so it deletes far more than it must and
    # its result is an outer bound, not an estimate.
    keep_dom = ~dup_dom_rows
    keep = ~(dup_dom_rows | dup_vec_rows)
    te_domclean = pd.concat([te[te.y == 1], te_be[keep_dom]])
    te_clean = pd.concat([te[te.y == 1], te_be[keep]])
    g_full, g_clean = strata_of(te), strata_of(te_clean)
    g_dom = strata_of(te_domclean)
    vn_full, vn_clean = vn_group(te["tld"]), vn_group(te_clean["tld"])
    vn_dom = vn_group(te_domclean["tld"])
    out = {
        "seed": seed,
        "n_benign_test_domains": n_dom_te,
        "n_benign_test_domains_shared": n_dom_shared,
        "share_domains": round(n_dom_shared / n_dom_te, 4),
        "n_benign_test_rows": int(len(te_be)),
        "n_benign_test_rows_domain_twin": int(dup_dom_rows.sum()),
        "share_rows": round(float(dup_dom_rows.mean()), 4),
        "n_benign_test_rows_vector_twin": int(dup_vec_rows.sum()),
        "share_vectors": round(float(dup_vec_rows.mean()), 4),
        "share_rows_any_twin": round(float((dup_dom_rows | dup_vec_rows).mean()), 4),
        "n_benign_after_domclean": int(keep_dom.sum()),
        "n_benign_after": int(keep.sum()),
    }
    y, s = te.y.to_numpy(int), te.pred.to_numpy(float)
    yc, sc = te_clean.y.to_numpy(int), te_clean.pred.to_numpy(float)
    yd, sd = te_domclean.y.to_numpy(int), te_domclean.pred.to_numpy(float)
    out["auc_vn_pooled"] = round(_auc(y[vn_full], s[vn_full]), 4)
    out["auc_other_pooled"] = round(_auc(y[~vn_full], s[~vn_full]), 4)
    out["auc_vn_pooled_domclean"] = round(_auc(yd[vn_dom], sd[vn_dom]), 4)
    out["auc_other_pooled_domclean"] = round(_auc(yd[~vn_dom], sd[~vn_dom]), 4)
    out["auc_vn_pooled_clean"] = round(_auc(yc[vn_clean], sc[vn_clean]), 4)
    out["auc_other_pooled_clean"] = round(_auc(yc[~vn_clean], sc[~vn_clean]), 4)
    for k in STRATA:
        mf, mc, md = ((g_full == k).to_numpy(), (g_clean == k).to_numpy(),
                      (g_dom == k).to_numpy())
        out["auc_" + k] = round(_auc(y[mf], s[mf]), 4)
        out["auc_" + k + "_domclean"] = round(_auc(yd[md], sd[md]), 4)
        out["auc_" + k + "_clean"] = round(_auc(yc[mc], sc[mc]), 4)
    out["gap_pooled"] = round(out["auc_vn_pooled"] - out["auc_other_pooled"], 4)
    out["gap_pooled_domclean"] = round(
        out["auc_vn_pooled_domclean"] - out["auc_other_pooled_domclean"], 4)
    out["gap_pooled_clean"] = round(out["auc_vn_pooled_clean"] - out["auc_other_pooled_clean"], 4)
    return [out]


def brand_locus(df: pd.DataFrame) -> pd.DataFrame:
    toks = json.load(open(BRANDS))["tokens"]
    toklist = sorted({t["token"].lower() for t in toks if len(t["token"]) >= 5})
    ph = df[(df.y == 1) & df.date.notna()].copy()
    ph["rdom_s"] = ph.rdom.astype(str).str.lower()
    ph["dom_s"] = ph.dom.astype(str).str.lower()
    ph["url_s"] = ph.url.astype(str).str.lower()

    def has(hay: str) -> bool:
        flat = strip_diacritics(hay)
        squash = re.sub(r"[-\d.]", "", flat)
        return any((t in flat) or (t in squash) for t in toklist)

    # One row per URL, deduplicated by (registrable domain, path) so a campaign that re-uses one
    # path across many hosts is not counted once per host.
    rows = []
    for _, r in ph.iterrows():
        rd, dom, url = r.rdom_s, r.dom_s, r.url_s
        sub = dom[: -len(rd)] if rd and dom.endswith(rd) else ""
        i = url.find(dom)
        tail = url[i + len(dom):] if i >= 0 else ""
        tail = tail.split("#", 1)[0]
        path = tail.split("?", 1)[0]
        rows.append({"rdom": rd, "path": path,
                     "in_rdom": int(has(rd)), "in_subdomain": int(has(sub)) if sub else 0,
                     "in_path": int(has(path)) if path.strip("/") else 0,
                     "has_path": int(bool(path.strip("/")))})
    out = pd.DataFrame(rows).drop_duplicates(subset=["rdom", "path"])
    return out


def run_suffix_blindspot(family='CatBoost', seeds=5, cal_frac=0.15, skip_shap=False):
    args = SimpleNamespace(family=family, seeds=seeds, cal_frac=cal_frac, skip_shap=skip_shap)
    df = load()
    feats = [c for c in COMPPHISH if c in df.columns]
    strata, aucs, thrs, leaks = [], [], [], []
    for seed in range(args.seeds):
        tr, m, cal, te = split_and_fit(df, feats, args.family, seed, args.cal_frac)
        strata += strata_rows(te, seed)
        aucs.append(auc_row(te, seed))
        thrs += threshold_rows(cal, te, seed)
        leaks += leakage_rows(tr, te, feats, seed)
        a = aucs[-1]
        print(f"[i] seed {seed}: AUC .vn pooled {a['auc_vn_pooled']:.3f} = "
              f"short {a['auc_vn_short']:.3f} + compound {a['auc_vn_compound']:.3f} "
              f"(cross-stratum pairs {a['cross_pair_share']:.3f}); "
              f"other {a['auc_other_pooled']:.3f}, cc_short {a['auc_cc_short']:.3f}")
    st = pd.DataFrame(strata)
    if args.family != "CatBoost":
        # Cross-family stress tests get their own compact artefacts. They must not overwrite the
        # canonical CatBoost SHAP, threshold, leakage or brand-locus files, none of which this run
        # is intended to recompute.
        slug = re.sub(r"\W+", "", args.family).lower()
        st_out = STRATA_OUT.replace(".csv", f"_{slug}.csv")
        auc_out = AUC_OUT.replace(".csv", f"_{slug}.csv")
        os.makedirs(os.path.dirname(st_out), exist_ok=True)
        st.to_csv(st_out, index=False)
        pd.DataFrame(aucs).to_csv(auc_out, index=False)
        print(f"[+] {st_out}")
        print(f"[+] {auc_out}")
        print("\n=== strata (mean over seeds) ===")
        print(st.groupby("stratum")[["n", "n_phish", "n_benign", "fnr", "fpr", "auc"]]
              .mean().reindex(list(STRATA)).round(4).to_string())
        return
    # (a) the signed tld_len attribution per stratum, on the SAME single-model split the per-TLD
    # would put a table's SHAP column and its FNR column on different models.
    if not args.skip_shap:
        import shap
        m0, _tr0, te0, _ph0 = split_fit(df, feats, args.family, 0)
        sv = shap.TreeExplainer(m0).shap_values(te0[feats].to_numpy(float))
        if isinstance(sv, list):
            sv = sv[1]
        te0 = te0.copy()
        te0["shap_tld_len"] = sv[:, feats.index("tld_len")]
        te0["pred"] = m0.predict_proba(te0[feats].to_numpy(float))[:, 1]
        g0 = strata_of(te0)
        srows = []
        for k in STRATA:
            sub = te0[(g0 == k).to_numpy()]
            gp = sub[sub.y == 1]
            srows.append({"stratum": k, "n": len(sub), "n_phish": len(gp),
                          "mean_shap_tld_len": round(float(sub.shap_tld_len.mean()), 5),
                          "fnr_single": round(float((gp.pred < 0.5).mean()), 4) if len(gp) else np.nan})
        sh = pd.DataFrame(srows)
        st = st.merge(sh, on="stratum", how="left", suffixes=("", "_shapsplit"))
        print("\n=== signed SHAP(tld_len) by stratum (single-model split, seed 0) ===")
        print(sh.to_string(index=False))
    os.makedirs(os.path.dirname(STRATA_OUT), exist_ok=True)
    st.to_csv(STRATA_OUT, index=False)
    pd.DataFrame(aucs).to_csv(AUC_OUT, index=False)
    pd.DataFrame(thrs).to_csv(THR_OUT, index=False)
    pd.DataFrame(leaks).to_csv(LEAK_OUT, index=False)
    lo = brand_locus(df)
    lo.to_csv(LOCUS_OUT, index=False)
    for p in (STRATA_OUT, AUC_OUT, THR_OUT, LEAK_OUT, LOCUS_OUT):
        print(f"[+] {p}")
    print("\n=== strata (mean over seeds) ===")
    print(st.groupby("stratum")[["n", "n_phish", "n_benign", "fnr", "fpr", "auc"]]
          .mean().reindex(list(STRATA)).round(4).to_string())
    print("\n=== decision rules (mean over seeds) ===")
    tf = pd.DataFrame(thrs)
    cols = ["fnr_vn_pooled", "fnr_vn_short", "fnr_cc_short", "fnr_vn_compound", "fnr_long",
            "fpr", "f1", "mcc", "bal_acc"]
    print(tf.groupby("rule")[cols].mean().round(4).to_string())
    print("\n=== leakage (mean over seeds) ===")
    lf = pd.DataFrame(leaks)
    print(lf[["share_domains", "share_rows", "share_vectors", "share_rows_any_twin",
              "gap_pooled", "gap_pooled_domclean", "gap_pooled_clean",
              "auc_vn_short", "auc_vn_short_domclean", "auc_long", "auc_long_domclean"]]
          .mean().round(4).to_string())
    print("\n=== brand locus (dated phishing, deduplicated by domain+path) ===")
    print(f"rows {len(lo)}; in registrable domain {int(lo.in_rdom.sum())} "
          f"({lo.in_rdom.mean():.4f}); in subdomain {int(lo.in_subdomain.sum())} "
          f"({lo.in_subdomain.mean():.4f}); with a non-empty path {int(lo.has_path.sum())}; "
          f"in path {int(lo.in_path.sum())} "
          f"({lo[lo.has_path == 1].in_path.mean() if lo.has_path.sum() else float('nan'):.4f} "
          f"of those)")

In [ ]:
run_suffix_blindspot()
run_suffix_blindspot(family="Stack[CB+LR]")
for f in ("p6_suffix_strata.csv", "p6_suffix_auc.csv", "p6_suffix_threshold.csv", "p6_leakage.csv", "p6_brand_locus.csv",
          "p6_suffix_strata_stackcblr.csv", "p6_suffix_auc_stackcblr.csv"):
    check_file("6 suffix strata", f"data/processed/p6/{f}")
report("6 suffix strata")

## Bước 7: Biên chi phí

Muốn mô hình bắt `.vn` tốt hơn thì phải trả bằng báo nhầm. Bước này vẽ toàn bộ biên đánh đổi đó: với tổng mức báo nhầm của corpus cố định, tỷ lệ bỏ sót
nhỏ nhất đạt được cho `.vn` và cho phần còn lại là bao nhiêu khi mỗi nhóm có ngưỡng riêng. Có hai phiên bản:
- "oracle": đặt ngưỡng trực tiếp trên tập kiểm, cho biết giới hạn tốt nhất có thể;
- "deployable": đặt ngưỡng trên tập hiệu chỉnh rồi đem áp lên tập kiểm, tức điều làm được khi triển khai.

Bốn quy tắc của Bước 4 được đánh dấu trên biên này. Bước này cũng tính mức báo nhầm của cả corpus cần có để cả hai nhóm cùng bỏ sót không quá một mức cho
trước (5%, 10%, 20%, 30%). Dùng lại đúng phép chia và mô hình của Bước 4 bằng cách gọi `split_and_fit`.

In [ ]:
TAUS_VN = [0.01, 0.05, 0.10, 0.20, 0.30, 0.50, 0.906]


TAUS_BOTH = [0.05, 0.10, 0.20, 0.30]


TAU_GRID = np.unique(np.round(np.concatenate([np.linspace(0.0, 1.0, 101), TAUS_VN]), 3))


def roc_arrays(y: np.ndarray, s: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y, s)
    return fpr, tpr


def min_fpr_for_miss(fpr: np.ndarray, tpr: np.ndarray, tau: float) -> float:
    i = int(np.searchsorted(tpr, 1.0 - tau - 1e-12, side="left"))
    if i >= len(tpr):                      # unreachable even at FPR 1 (cannot happen here)
        return 1.0
    return float(fpr[i])


def min_miss_for_fpr(fpr: np.ndarray, tpr: np.ndarray, budget: float) -> float:
    if budget < 0:
        return 1.0                          # the other group already overspent the whole budget
    i = int(np.searchsorted(fpr, budget + 1e-12, side="right")) - 1
    if i < 0:
        return 1.0
    return float(1.0 - tpr[i])


def frontier_curve(roc_vn, roc_ot, pi_vn: float, b: float,
                   taus: np.ndarray) -> np.ndarray:
    pi_ot = 1.0 - pi_vn
    out = np.empty(len(taus))
    for k, tau in enumerate(taus):
        f_vn = min_fpr_for_miss(roc_vn[0], roc_vn[1], float(tau))
        rem = (b - pi_vn * f_vn) / pi_ot if pi_ot > 0 else -1.0
        out[k] = min_miss_for_fpr(roc_ot[0], roc_ot[1], rem)
    return out


def min_budget_for_both(roc_vn, roc_ot, pi_vn: float, tau: float) -> float:
    f_vn = min_fpr_for_miss(roc_vn[0], roc_vn[1], tau)
    f_ot = min_fpr_for_miss(roc_ot[0], roc_ot[1], tau)
    return float(pi_vn * f_vn + (1.0 - pi_vn) * f_ot)


def is_pareto_frontier(miss_vn: np.ndarray, miss_ot: np.ndarray) -> bool:
    for i in range(len(miss_vn)):
        for j in range(len(miss_vn)):
            if i == j:
                continue
            if (miss_vn[j] <= miss_vn[i] + 1e-12 and miss_ot[j] <= miss_ot[i] + 1e-12
                    and (miss_vn[j] < miss_vn[i] - 1e-12 or miss_ot[j] < miss_ot[i] - 1e-12)):
                return False
    return True


def bf_one_seed(df, feats, family: str, seed: int, cal_frac: float):
    _tr, _m, cal, te = split_and_fit(df, feats, family, seed, cal_frac)
    cal_be = cal[cal.y == 0]
    thr, alpha = threshold_rules(cal_be)

    y = te.y.to_numpy(int)
    s = te.pred.to_numpy(float)
    g = te.vn.to_numpy(bool)
    be = y == 0
    pi_vn = float((g & be).sum() / be.sum())        # the group's share of the BENIGN pool

    roc_vn = roc_arrays(y[g], s[g])
    roc_ot = roc_arrays(y[~g], s[~g])

    # the deployed budget: the 0.5 rule's own realised corpus FPR on this seed's test set
    b0 = float((s[be] >= 0.5).mean())

    # ---- the four published rules, at the coordinates the published table reports
    pts = []
    for cond, t in thr.items():
        pred = s >= np.where(g, t["vn"], t["other"])
        pts.append({
            "seed": seed, "condition": cond,
            "miss_vn": float((pred[g & (y == 1)] == 0).mean()),
            "miss_other": float((pred[~g & (y == 1)] == 0).mean()),
            "fpr_vn": float((pred[g & be] == 1).mean()),
            "fpr_other": float((pred[~g & be] == 1).mean()),
            "budget": float((pred[be] == 1).mean()),
            "thr_vn": t["vn"], "thr_other": t["other"],
        })
    ppts = pd.DataFrame(pts)
    b_vnonly = float(ppts[ppts.condition == "vn-only"].budget.iloc[0])

    # ---- ORACLE frontier at each budget level (thresholds read off the test curves: an upper
    # bound on what any allocation of that budget can achieve, deployable or not)
    budgets = {"half": 0.5 * b0, "deployed": b0, "double": 2.0 * b0, "vn-only": b_vnonly}
    fr = []
    for bname, b in budgets.items():
        miss_ot = frontier_curve(roc_vn, roc_ot, pi_vn, b, TAU_GRID)
        for tau, mo in zip(TAU_GRID, miss_ot):
            fr.append({"seed": seed, "budget_name": bname, "budget": b,
                       "tau_vn": float(tau), "miss_other": float(mo), "pi_vn": pi_vn})
    frontier = pd.DataFrame(fr)

    # ---- DEPLOYABLE frontier at the deployed budget: thresholds are calibration-benign
    # quantiles, the same instrument the published rules use, so this curve is reachable. The
    # budget is booked in CALIBRATION units (all a deployer can observe); the realised test
    # budget is recorded next to it so the drift is visible rather than assumed away.
    cal_vn = cal_be[cal_be.vn].pred.to_numpy(float)
    cal_ot = cal_be[~cal_be.vn].pred.to_numpy(float)
    pi_vn_cal = float(len(cal_vn) / len(cal_be))
    cal_rows = []
    for a_vn in np.round(np.linspace(0.0, 1.0, 101), 3):
        a_ot = (b0 - pi_vn_cal * a_vn) / (1.0 - pi_vn_cal)
        if a_ot < 0:
            continue
        t_vn = float(np.quantile(cal_vn, 1.0 - a_vn)) if len(cal_vn) else 0.5
        t_ot = float(np.quantile(cal_ot, 1.0 - min(a_ot, 1.0))) if len(cal_ot) else 0.5
        pred = s >= np.where(g, t_vn, t_ot)
        cal_rows.append({
            "seed": seed, "alpha_vn": float(a_vn), "alpha_other": float(a_ot),
            "thr_vn": t_vn, "thr_other": t_ot,
            "miss_vn": float((pred[g & (y == 1)] == 0).mean()),
            "miss_other": float((pred[~g & (y == 1)] == 0).mean()),
            "budget_test": float((pred[be] == 1).mean()),
        })
    caldf = pd.DataFrame(cal_rows)

    # ---- price of parity
    par = pd.DataFrame([{"seed": seed, "tau": t,
                         "b_star": min_budget_for_both(roc_vn, roc_ot, pi_vn, t),
                         "b0": b0}
                        for t in TAUS_BOTH])
    return frontier, caldf, ppts, par, b0, pi_vn


CSV_NAMES = ("p6_budget_frontier", "p6_budget_frontier_cal", "p6_budget_frontier_points",
             "p6_budget_parity")


def run_budget_frontier(family='CatBoost', seeds=5, cal_frac=0.15):
    args = SimpleNamespace(family=family, seeds=seeds, cal_frac=cal_frac)
    df = load()
    feats = [c for c in COMPPHISH if c in df.columns]
    FR, CAL, PTS, PAR = [], [], [], []
    for s in range(args.seeds):
        fr, cal, pts, par, b0, pi = bf_one_seed(df, feats, args.family, s, args.cal_frac)
        FR.append(fr)
        CAL.append(cal)
        PTS.append(pts)
        PAR.append(par)
        print(f"[i] seed {s}: budget(deployed FPR)={b0:.3f} pi_vn={pi:.3f} "
              f"b*(both<=0.10)={par[np.isclose(par.tau, 0.10)].b_star.iloc[0]:.3f}")
    fr = pd.concat(FR, ignore_index=True)
    cal = pd.concat(CAL, ignore_index=True)
    pts = pd.concat(PTS, ignore_index=True)
    par = pd.concat(PAR, ignore_index=True)
    seeds = args.seeds
    os.makedirs(os.path.join(PROC, "p6"), exist_ok=True)
    for name, d in zip(CSV_NAMES, (fr, cal, pts, par)):
        d.to_csv(os.path.join(PROC, "p6", name + ".csv"), index=False)
        print(f"[+] data/processed/p6/{name}.csv")

In [ ]:
run_budget_frontier()
for f in ("p6_budget_frontier.csv", "p6_budget_frontier_cal.csv", "p6_budget_frontier_points.csv", "p6_budget_parity.csv"):
    check_file("7 budget frontier", f"data/processed/p6/{f}")
report("7 budget frontier")

## Bước 8: SHAP trôi theo thời gian

Một đề xuất vận hành hay gặp: theo dõi SHAP của mô hình đang chạy để phát hiện khi nó bắt đầu kém đi. Bước này thử đề xuất đó. Mô hình được huấn luyện trên
30% tin phishing cũ nhất cộng toàn bộ benign, rồi chấm sáu cửa sổ thời gian kế tiếp, mỗi cửa sổ cùng số tin phishing. Với mỗi cửa sổ: tỷ lệ bắt được (chỉ
có phishing nên chính là recall) và |SHAP| trung bình của từng đặc trưng. Hệ số Spearman giữa recall và |SHAP| qua các cửa sổ chỉ là mô tả hướng và độ lớn,
không phải kiểm định, vì chỉ có sáu cửa sổ.

In [ ]:
AD_OUT = "data/processed/p6/p6_attribution_drift.csv"


KEY = ["dot_cnt", "dash_cnt", "tld_len"]


def shap_profile(model, X, feats, sample, rng):
    import shap
    sub = X if len(X) <= sample else X[rng.choice(len(X), size=sample, replace=False)]
    sv = shap.TreeExplainer(model).shap_values(sub)
    if isinstance(sv, list):
        sv = sv[1]
    return pd.Series(np.abs(sv).mean(axis=0), index=feats)


def run_attribution_drift(family='CatBoost', train_frac=0.3, windows=6, sample=2000, seed=0):
    args = SimpleNamespace(family=family, train_frac=train_frac, windows=windows, sample=sample, seed=seed)
    df = load()
    feats = [c for c in COMPPHISH if c in df.columns]
    ph = df[(df.y == 1) & df.date.notna()].sort_values("date").reset_index(drop=True)
    be = df[df.y == 0]
    rng = np.random.RandomState(args.seed)
    # deployed model: earliest train-frac of dated phishing + the (stationary) benign class
    cut = int(len(ph) * args.train_frac)
    ph_tr, ph_fwd = ph.iloc[:cut], ph.iloc[cut:]
    Xtr = pd.concat([ph_tr[feats], be[feats]]).to_numpy(float)
    ytr = np.r_[np.ones(len(ph_tr), int), np.zeros(len(be), int)]
    model = make_any_model(args.family, args.seed)
    model.fit(Xtr, ytr)
    print(f"[i] deployed {args.family}: train on {len(ph_tr)} phishing "
          f"(<= {ph_tr.date.max().date()}) + {len(be)} benign; "
          f"forward on {len(ph_fwd)} phishing in {args.windows} windows")
    # equal-count forward windows
    ph_fwd = ph_fwd.reset_index(drop=True)
    bins = np.array_split(np.arange(len(ph_fwd)), args.windows)
    rows = []
    for w, idx in enumerate(bins):
        win = ph_fwd.iloc[idx]
        X = win[feats].to_numpy(float)
        recall = float((model.predict(X) == 1).mean())          # phishing-only -> recall = catch rate
        prof = shap_profile(model, X, feats, args.sample, rng)
        row = {"window": w, "n": len(win),
               "date_start": win.date.min().date().isoformat(),
               "date_end": win.date.max().date().isoformat(),
               "date_mid": win.date.iloc[len(win) // 2].date().isoformat(),
               "recall": round(recall, 4)}
        for f in feats:
            row[f"shap_{f}"] = round(float(prof[f]), 5)
        rows.append(row)
        print(f"  w{w} {row['date_start']}..{row['date_end']} n={len(win)} "
              f"recall={recall:.3f} " + " ".join(f"{k}={prof[k]:.3f}" for k in KEY))
    out = pd.DataFrame(rows)
    # descriptive co-variation: Spearman rho between recall and each key feature's |SHAP| over
    # windows. Reported as direction + magnitude, NOT as a significance test (too few windows).
    from scipy.stats import spearmanr
    corr = {}
    for f in KEY:
        r, _ = spearmanr(out["recall"], out[f"shap_{f}"])
        corr[f] = round(float(r), 3)
    out.attrs["recall_shap_spearman"] = corr
    os.makedirs(os.path.dirname(AD_OUT), exist_ok=True)
    out.to_csv(AD_OUT, index=False)
    # stash the correlation as a trailing comment row-free sidecar the asset generator reads
    with open(AD_OUT.replace(".csv", "_corr.csv"), "w", encoding="utf-8") as f:
        f.write("feature,recall_shap_spearman\n")
        for k, v in corr.items():
            f.write(f"{k},{v}\n")
    print(f"[+] {AD_OUT}")
    print(f"[i] Spearman rho(recall, |SHAP|) over windows: {corr}")
    print(f"[i] recall drift: {out.recall.iloc[0]:.3f} -> {out.recall.iloc[-1]:.3f}")

In [ ]:
run_attribution_drift()
for f in ("p6_attribution_drift.csv", "p6_attribution_drift_corr.csv"):
    check_file("8 attribution drift", f"data/processed/p6/{f}")
report("8 attribution drift")

## Bước 9: Các ca cụ thể

Bảng ca cụ thể của bài: bốn tin `.vn` bị bỏ sót có `tld_len` đẩy về phía benign mạnh nhất, một tin phishing không phải `.vn` được bắt, và một tin `.vn` được
bắt (nếu có). Với mỗi tin: điểm của mô hình, SHAP của `tld_len`, và ba đặc trưng có |SHAP| lớn nhất kèm dấu. Mỗi tên miền đăng ký chỉ xuất hiện một lần.

In [ ]:
CS_OUT = "data/processed/p6/p6_case_studies.csv"


N_MISSED = 4      # missed .vn phishing examples


N_CAUGHT = 2      # caught contrasts (one .vn if any is caught, one non-.vn)


def top_signed(sv_row, feats, k=3):
    order = np.argsort(-np.abs(sv_row))[:k]
    return "; ".join(f"{feats[i]}:{sv_row[i]:+.2f}" for i in order)


def run_case_studies(family='CatBoost', seed=0):
    args = SimpleNamespace(family=family, seed=seed)
    df = load()
    feats = [c for c in COMPPHISH if c in df.columns]
    ph = df[(df.y == 1) & df.date.notna()].sort_values("date").reset_index(drop=True)
    be = df[df.y == 0].reset_index(drop=True)
    cut = int(len(ph) * 0.70)
    ph_tr, ph_te = ph.iloc[:cut], ph.iloc[cut:]
    ph_te = ph_te[~ph_te.rdom.isin(set(ph_tr.rdom))]
    rng = np.random.RandomState(args.seed)
    bmask = rng.rand(len(be)) < 0.70
    tr = pd.concat([ph_tr, be[bmask]])
    te = pd.concat([ph_te, be[~bmask]]).reset_index(drop=True)
    m = make_any_model(args.family, args.seed)
    m.fit(tr[feats].to_numpy(float), tr["y"].to_numpy(int))
    import shap
    sv = shap.TreeExplainer(m).shap_values(te[feats].to_numpy(float))
    if isinstance(sv, list):
        sv = sv[1]
    te = te.reset_index(drop=True)
    te["pred"] = m.predict_proba(te[feats].to_numpy(float))[:, 1]
    te["tld_str"] = te["tld"].astype(str).str.lower()
    te["is_vn"] = te.tld_str.str.endswith("vn")
    i_tld = feats.index("tld_len")
    te["shap_tld_len"] = sv[:, i_tld]
    def pick(mask, sort_col, ascending, n):
        sub = te[mask].sort_values(sort_col, ascending=ascending)
        sub = sub.drop_duplicates("rdom")
        return sub.head(n)
    # missed .vn phishing: most benign-ward tld_len contribution among the false negatives
    missed = pick((te.y == 1) & te.is_vn & (te.pred < 0.5), "shap_tld_len", True, N_MISSED)
    # caught contrasts: a caught non-.vn phishing (tld_len should push phishing-ward), and a caught
    # .vn phishing if the model got any right (shows the prior is beatable when other signal is loud)
    caught_other = pick((te.y == 1) & ~te.is_vn & (te.pred >= 0.5), "pred", False, 1)
    caught_vn = pick((te.y == 1) & te.is_vn & (te.pred >= 0.5), "pred", False, 1)
    rows = []
    for kind, sub in [("missed .vn", missed), ("caught non-.vn", caught_other),
                      ("caught .vn", caught_vn)]:
        for pos, r in sub.iterrows():
            rows.append({
                "kind": kind,
                "domain": str(r.get("rdom") or r["url"]),
                "score": round(float(r["pred"]), 3),
                "shap_tld_len": round(float(r["shap_tld_len"]), 3),
                "top_signed_shap": top_signed(sv[pos], feats),
            })
    out = pd.DataFrame(rows)
    os.makedirs(os.path.dirname(CS_OUT), exist_ok=True)
    out.to_csv(CS_OUT, index=False)
    n_vn_miss = int(((te.y == 1) & te.is_vn & (te.pred < 0.5)).sum())
    n_vn = int(((te.y == 1) & te.is_vn).sum())
    print(f"[i] .vn phishing in test: {n_vn}, missed {n_vn_miss} (FNR {n_vn_miss / n_vn:.3f})")
    print(out.to_string(index=False))
    print(f"[+] {CS_OUT}")

In [ ]:
run_case_studies()
check_file("9 case studies", "data/processed/p6/p6_case_studies.csv")
report("9 case studies")

## Bước 10: CharCNN đọc ký tự URL

Điểm mù có phải do 21 đặc trưng đếm quá thô không? Bước này huấn luyện một mạng tích chập trên chính chuỗi ký tự của URL (CharCNN) rồi đo tỷ lệ bỏ sót theo
bốn tầng đuôi, cạnh CatBoost trên cùng phép chia. Mạng: mỗi ký tự thành một vector 32 chiều, bốn bộ lọc tích chập độ rộng 3 đến 6 (128 kênh mỗi bộ), lấy giá
trị lớn nhất theo chiều dài, dropout 0,3, một lớp ra. URL cắt ở 128 ký tự. Huấn luyện tối đa 8 epoch với Adam (learning rate 1e-3, batch 256), dừng sớm khi
loss trên 10% tập huấn luyện giữ lại không giảm sau 2 epoch, rồi lấy lại trạng thái tốt nhất. Bảng ký tự chỉ lấy từ tập huấn luyện; ký tự lạ thành một mã
riêng. Torch được đặt ở chế độ tất định nên chạy lại cho cùng kết quả.

Bước này chạy 5 seed và mất khoảng 30 phút trên CPU.

In [ ]:
MAX_LEN = 128          # p99 is 43; 20 rows of 53,116 are longer than this and are truncated


EMB = 32


WIDTHS = (3, 4, 5, 6)


FILTERS = 128


DROPOUT = 0.3


BATCH = 256


LR = 1e-3


VAL_FRAC = 0.10        # carved from TRAIN, never from the test window


def encode(urls, vocab):
    out = np.zeros((len(urls), MAX_LEN), dtype=np.int64)
    for i, u in enumerate(urls):
        for j, ch in enumerate(str(u)[:MAX_LEN]):
            out[i, j] = vocab.get(ch, 1)
    return out


def build_vocab(urls):
    chars = sorted({ch for u in urls for ch in str(u)[:MAX_LEN]})
    return {ch: i + 2 for i, ch in enumerate(chars)}


def make_net(vocab_size, seed):
    import torch, torch.nn as nn
    torch.manual_seed(seed)

    class Net(nn.Module):
        def __init__(self):
            super().__init__()
            self.emb = nn.Embedding(vocab_size + 2, EMB, padding_idx=0)
            self.convs = nn.ModuleList(
                [nn.Conv1d(EMB, FILTERS, w, padding=w // 2) for w in WIDTHS])
            self.drop = nn.Dropout(DROPOUT)
            self.fc = nn.Linear(FILTERS * len(WIDTHS), 1)

        def forward(self, x):
            h = self.emb(x).transpose(1, 2)
            h = torch.cat([c(h).relu().amax(dim=2) for c in self.convs], dim=1)
            return self.fc(self.drop(h)).squeeze(1)

    return Net()


def fit_predict(tr_urls, ytr, te_urls, seed, epochs, patience=2, device="cpu"):
    import torch
    from torch.utils.data import DataLoader, TensorDataset
    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)

    rng = np.random.RandomState(seed)
    vmask = rng.rand(len(tr_urls)) < VAL_FRAC
    vocab = build_vocab(np.asarray(tr_urls)[~vmask])          # TRAIN-only alphabet

    Xtr = torch.from_numpy(encode(np.asarray(tr_urls)[~vmask], vocab))
    Xva = torch.from_numpy(encode(np.asarray(tr_urls)[vmask], vocab))
    Xte = torch.from_numpy(encode(np.asarray(te_urls), vocab))
    ytr_t = torch.from_numpy(np.asarray(ytr)[~vmask].astype(np.float32))
    yva_t = torch.from_numpy(np.asarray(ytr)[vmask].astype(np.float32))

    net = make_net(len(vocab), seed).to(device)
    Xva, yva_t = Xva.to(device), yva_t.to(device)
    opt = torch.optim.Adam(net.parameters(), lr=LR)
    lossf = torch.nn.BCEWithLogitsLoss()
    loader = DataLoader(TensorDataset(Xtr, ytr_t), batch_size=BATCH, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))

    best, best_state, bad = float("inf"), None, 0
    for _ in range(epochs):
        net.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            lossf(net(xb), yb).backward()
            opt.step()
        net.eval()
        with torch.no_grad():
            vloss = lossf(net(Xva), yva_t).item()
        if vloss < best - 1e-4:
            best, bad = vloss, 0
            best_state = {k: v.clone() for k, v in net.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    if best_state is not None:
        net.load_state_dict(best_state)
    net.eval()
    with torch.no_grad():
        # score in slices: the test corpus can be 53k rows and the whole tensor need not sit on
        # the accelerator at once
        out = [torch.sigmoid(net(Xte[i:i + 4096].to(device))).cpu()
               for i in range(0, len(Xte), 4096)]
        return torch.cat(out).numpy()


def rates(te: pd.DataFrame, score: np.ndarray, tag: str, seed: int) -> list[dict]:
    st = strata_of(te)
    y = te["y"].to_numpy(int)
    pred = (score >= 0.5).astype(int)
    out = []
    for name in STRATA:
        m = (st == name).to_numpy()
        ph = m & (y == 1)
        if ph.sum() == 0:
            continue
        out.append({"model": tag, "seed": seed, "stratum": name,
                    "n_phish": int(ph.sum()),
                    "FNR": float((pred[ph] == 0).mean()),
                    "n_benign": int((m & (y == 0)).sum())})
    return out


def run_charcnn_strata(seeds=5, epochs=8, cut=0.7, reference='CatBoost', out='data/processed/p6/p6_charcnn_strata.csv'):
    a = SimpleNamespace(seeds=seeds, epochs=epochs, cut=cut, reference=reference, out=out)
    os.chdir(ROOT)
    df = load()
    feats = [c for c in COMPPHISH if c in df.columns]
    ph = df[(df.y == 1) & df.date.notna()].sort_values("date").reset_index(drop=True)
    be = df[df.y == 0].reset_index(drop=True)
    ph_tr, ph_te, _ = split_phishing(ph, a.cut)
    rows = []
    for s in range(a.seeds):
        rng = np.random.RandomState(s)
        bmask = rng.rand(len(be)) < a.cut
        tr = pd.concat([ph_tr, be[bmask]])
        te = pd.concat([ph_te, be[~bmask]])

        t0 = time.time()
        sc = fit_predict(tr["url"].tolist(), tr["y"].to_numpy(int), te["url"].tolist(), s, a.epochs)
        rows += rates(te, sc, "CharCNN", s)
        print(f"  CharCNN  seed={s} ({time.time()-t0:.0f}s)")

        m = make_any_model(a.reference, s)
        m.fit(tr[feats].to_numpy(float), tr["y"].to_numpy(int))
        rows += rates(te, m.predict_proba(te[feats].to_numpy(float))[:, 1], a.reference, s)
        print(f"  {a.reference:<8} seed={s}")
    out = pd.DataFrame(rows)
    os.makedirs(os.path.dirname(a.out), exist_ok=True)
    out.to_csv(a.out, index=False)
    print("\n  per-stratum FNR at tau=0.5, mean over seeds:")
    piv = out.pivot_table(index="stratum", columns="model", values="FNR", aggfunc="mean")
    n = out.groupby("stratum")["n_phish"].mean().astype(int)
    for st in STRATA:
        if st in piv.index:
            print(f"    {st:<13} n={n[st]:>5}  " +
                  "  ".join(f"{c} {piv.loc[st, c]:.3f}" for c in piv.columns))
    print(f"[+] {a.out}")
    return 0

In [ ]:
t0 = time.time()
run_charcnn_strata()
print(f"{time.time() - t0:.0f}s")
check_file("10 charcnn", "data/processed/p6/p6_charcnn_strata.csv")
report("10 charcnn")

## Bước 11: Ablation tiến cứu

Bài đọc `tld_len` như một "giá trị tiên nghiệm theo đuôi". Bước này kiểm bằng can thiệp thật thay vì chỉ đọc SHAP: bỏ hẳn đặc trưng rồi huấn luyện lại. Ba
cánh: đủ 21 đặc trưng; bỏ `tld_len`; bỏ cả họ đặc trưng độ dài (`tld_len`, `url_len`, `dom_len`).

Phép chia ở đây độc lập với các bước trước và theo lịch: 60% tin phishing đầu để huấn luyện, 20% tiếp theo để hiệu chỉnh, 20% cuối để kiểm, cắt tại ngày (một
ngày không nằm ở hai cửa sổ). Tin phishing kiểm có tên miền đã xuất hiện trước đó bị bỏ. Tin benign chia theo tên miền đăng ký bằng một hàm băm SHA-256 cố
định (60/20/20), nên một tên miền benign không xuất hiện ở hai phía và phép chia giống nhau trên mọi máy. Ngưỡng của mỗi cánh đặt ở cùng mức báo nhầm trên
tập hiệu chỉnh.

In [ ]:
PA_OUT = os.path.join(ROOT, "data", "processed", "p6", "p6_prospective_ablation.csv")


PA_PROTO = os.path.join(ROOT, "data", "processed", "p6", "p6_prospective_protocol.csv")


ARMS = {
    "full": (),
    "minus_tld_len": ("tld_len",),
    "minus_length_family": ("tld_len", "url_len", "dom_len"),
}


def stable_fold(value: str) -> int:
    digest = hashlib.sha256(str(value).encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "big") % 100


def stratum(frame: pd.DataFrame) -> pd.Series:
    tld = frame["tld"].astype(str).str.lower().str.lstrip(".")
    vn = tld.str.endswith("vn")
    short = tld.str.len().eq(2)
    return pd.Series(np.where(short, np.where(vn, "vn_short", "cc_short"),
                              np.where(vn, "vn_compound", "long")), index=frame.index)


def pa_split(df: pd.DataFrame):
    ph = df[(df.y == 1) & df.date.notna()].sort_values(["date", "url"]).reset_index(drop=True)
    n = len(ph)
    k_fit, k_cal = int(0.60 * n), int(0.80 * n)
    # Pick calendar boundaries at the target quantiles and keep a boundary date wholly on the
    # earlier side. Positional slicing would let one day occur in two windows when dates tie.
    fit_end, cal_end = ph.iloc[k_fit - 1].date, ph.iloc[k_cal - 1].date
    ph_fit = ph[ph.date <= fit_end]
    ph_cal = ph[(ph.date > fit_end) & (ph.date <= cal_end)]
    ph_test_raw = ph[ph.date > cal_end]
    prior_domains = set(pd.concat([ph_fit.rdom, ph_cal.rdom]).astype(str))
    repeated = ph_test_raw.rdom.astype(str).isin(prior_domains)
    ph_test = ph_test_raw[~repeated]

    be = df[df.y == 0].copy()
    # Domain, rather than row order, URL, or Python's salted hash, makes the partition reproducible
    # and prevents a benign registrable domain from recurring across windows.
    fold = be["rdom"].astype(str).map(stable_fold)
    be_fit, be_cal, be_test = be[fold < 60], be[(fold >= 60) & (fold < 80)], be[fold >= 80]
    return ph_fit, ph_cal, ph_test, be_fit, be_cal, be_test, int(repeated.sum()), n


def safe_auc(y, score) -> float:
    return float(roc_auc_score(y, score)) if len(np.unique(y)) > 1 else float("nan")


def pa_metrics(y, score, threshold) -> dict[str, float]:
    y, score = np.asarray(y, int), np.asarray(score, float)
    pred = score >= threshold
    return {
        "n": len(y), "n_phish": int(y.sum()), "n_benign": int((y == 0).sum()),
        "fnr": float(1 - pred[y == 1].mean()) if (y == 1).any() else np.nan,
        "fpr": float(pred[y == 0].mean()) if (y == 0).any() else np.nan,
        "roc_auc": safe_auc(y, score),
        "pr_auc": float(average_precision_score(y, score)) if (y == 1).any() else np.nan,
        "f1": float(f1_score(y, pred, zero_division=0)),
        "mcc": float(matthews_corrcoef(y, pred)) if len(np.unique(pred)) > 1 else 0.0,
        "balanced_accuracy": float(balanced_accuracy_score(y, pred)),
    }


def run_prospective_ablation(family='CatBoost', seeds=5, out='data/processed/p6/p6_prospective_ablation.csv'):
    args = SimpleNamespace(family=family, seeds=seeds, out=out)
    df = load()
    all_features = [c for c in COMPPHISH if c in df.columns]
    ph_fit, ph_cal, ph_test, be_fit, be_cal, be_test, n_repeat, n_dated = pa_split(df)
    fit = pd.concat([ph_fit, be_fit]).reset_index(drop=True)
    cal = pd.concat([ph_cal, be_cal]).reset_index(drop=True)
    test = pd.concat([ph_test, be_test]).reset_index(drop=True)
    test["stratum"] = stratum(test)
    protocol = pd.DataFrame([{
        "family": args.family,
        "phishing_fit_fraction": 0.60,
        "phishing_calibration_fraction": 0.20,
        "phishing_holdout_fraction": 0.20,
        "fit_start": ph_fit.date.min().date(), "fit_end": ph_fit.date.max().date(),
        "calibration_start": ph_cal.date.min().date(),
        "calibration_end": ph_cal.date.max().date(),
        "holdout_start": ph_test.date.min().date(), "holdout_end": ph_test.date.max().date(),
        "n_dated_phishing": n_dated, "n_fit_phishing": len(ph_fit),
        "n_calibration_phishing": len(ph_cal), "n_holdout_phishing": len(ph_test),
        "n_holdout_repeated_domains_removed": n_repeat,
        "n_fit_benign": len(be_fit), "n_calibration_benign": len(be_cal),
        "n_holdout_benign": len(be_test),
        "benign_partition": "stable SHA-256 registrable-domain hash; benign timestamps incomplete",
        "status": "locked forward (prospective-style), not a newly accrued prospective cohort",
    }])
    os.makedirs(os.path.dirname(args.out), exist_ok=True)
    protocol.to_csv(PA_PROTO, index=False)
    rows = []
    for seed in range(args.seeds):
        # Fix the operating budget before looking at any ablation's holdout scores: it is the
        # full model's calibration-benign FPR at the historical 0.5 threshold.
        full = make_any_model(args.family, seed)
        full.fit(fit[all_features].to_numpy(float), fit.y.to_numpy(int))
        full_cal = full.predict_proba(cal[all_features].to_numpy(float))[:, 1]
        alpha = float((full_cal[cal.y.to_numpy(int) == 0] >= 0.5).mean())
        q = 1 - alpha

        for arm, removed in ARMS.items():
            features = [f for f in all_features if f not in removed]
            model = full if arm == "full" else make_any_model(args.family, seed)
            if arm != "full":
                model.fit(fit[features].to_numpy(float), fit.y.to_numpy(int))
            cal_score = (full_cal if arm == "full" else
                         model.predict_proba(cal[features].to_numpy(float))[:, 1])
            cal_be_score = cal_score[cal.y.to_numpy(int) == 0]
            threshold = float(np.quantile(cal_be_score, q))
            score = model.predict_proba(test[features].to_numpy(float))[:, 1]

            for group in ("overall",) + STRATA + ("short_all",):
                if group == "overall":
                    mask = np.ones(len(test), dtype=bool)
                elif group == "short_all":
                    mask = test.stratum.isin(["vn_short", "cc_short"]).to_numpy()
                else:
                    mask = test.stratum.eq(group).to_numpy()
                result = pa_metrics(test.y.to_numpy(int)[mask], score[mask], threshold)
                rows.append({
                    "seed": seed, "family": args.family, "arm": arm,
                    "removed_features": ";".join(removed) or "none",
                    "n_features": len(features), "stratum": group,
                    "alpha_calibration": alpha, "threshold": threshold, **result,
                })
        print(f"[i] seed={seed} alpha={alpha:.4f} threshold arms complete", flush=True)
    out = pd.DataFrame(rows)
    out.to_csv(args.out, index=False, float_format="%.6f")
    show = (out.groupby(["arm", "stratum"])[["fnr", "fpr", "roc_auc", "mcc"]]
               .mean().round(4))
    print(protocol.to_string(index=False))
    print(show.to_string())
    print(f"[+] {args.out}\n[+] {PA_PROTO}")

In [ ]:
run_prospective_ablation()
for f in ("p6_prospective_ablation.csv", "p6_prospective_protocol.csv"):
    check_file("11 prospective ablation", f"data/processed/p6/{f}")
report("11 prospective ablation")

## Bước 12: Độ nhạy theo tầng nhãn, và corpus một-dòng-mỗi-URL

### 12a. Tầng nhãn

Kiểm toán corpus ước tính 12,1% tin "phishing" lấy mẫu thực ra là trang hợp lệ, phần lớn ở tầng nguồn `bronze`. Bước này đo tỷ lệ bỏ sót theo tầng nguồn
(`bronze`, `silver`, `gold`) trong hai tầng đuôi ngắn (`.vn` trần và đuôi quốc gia 2 ký tự khác), trên phép chia của Bước 11, 5 seed CatBoost. Ô thưa được
giữ nguyên chứ không gộp; tầng nguồn và thời gian vẫn gắn với nhau nên đây là phân tích độ nhạy, không phải ước lượng nhân quả.

In [ ]:
SEEDS = 5


def _save(df, relative):
    path = os.path.join(ROOT, relative)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_csv(path, index=False, float_format="%.6f")
    print(f"[+] {path}")


def run_tier_sensitivity(df):
    feats = [c for c in COMPPHISH if c in df.columns]
    ph_fit, ph_cal, ph_test, be_fit, be_cal, be_test, _, _ = pa_split(df)
    fit = pd.concat([ph_fit, be_fit]).reset_index(drop=True)
    cal = pd.concat([ph_cal, be_cal]).reset_index(drop=True)
    test = pd.concat([ph_test, be_test]).reset_index(drop=True)
    test["suffix_stratum"] = stratum(test)
    records = []
    for seed in range(SEEDS):
        model = make_any_model("CatBoost", seed)
        model.fit(fit[feats].to_numpy(float), fit.y.to_numpy(int))
        cal_score = model.predict_proba(cal[feats].to_numpy(float))[:, 1]
        cal_be = cal_score[cal.y.to_numpy(int) == 0]
        alpha = float((cal_be >= 0.5).mean())
        threshold = float(np.quantile(cal_be, 1 - alpha))
        pred = model.predict_proba(test[feats].to_numpy(float))[:, 1] >= threshold
        for tier in ("bronze", "silver", "gold"):
            for suffix in ("vn_short", "cc_short"):
                mask = ((test.y.to_numpy(int) == 1) & test.tier.astype(str).eq(tier).to_numpy()
                        & test.suffix_stratum.eq(suffix).to_numpy())
                miss = int((~pred[mask]).sum())
                n = int(mask.sum())
                lo, hi = wilson(miss, n) if n else (np.nan, np.nan)
                records.append({"seed": seed, "tier": tier, "suffix_stratum": suffix,
                                "n": n, "misses": miss, "fnr": miss / n if n else np.nan,
                                "ci_low": lo, "ci_high": hi, "threshold": threshold})
    out = pd.DataFrame(records)
    _save(out, "data/processed/p6/p6_tier_sensitivity.csv")
    print(out.groupby(["tier", "suffix_stratum"])[["n", "fnr"]].mean().round(3))

In [ ]:
run_tier_sensitivity(load())
check_file("12 tier + dedup", "data/processed/p6/p6_tier_sensitivity.csv")

### 12b. Corpus một-dòng-mỗi-URL

Corpus đã công bố ghi 16.855 URL hai lần: bản có `www.` không ngày và bản không `www.` có ngày (cùng một host từ hai nguồn). Bước này dựng corpus chỉ giữ một
dòng cho mỗi host: bỏ phần `scheme://` và `www.` ở đầu, ưu tiên dòng có ngày, rồi dòng không có `www.`, rồi theo thứ tự dòng. Kết quả còn 36.261 dòng. Bảng đặc
trưng được lấy theo đúng các dòng giữ lại, không tính lại.

Sau đó chạy lại ba thí nghiệm trên corpus này (đo theo tầng đuôi không kèm SHAP, ngưỡng theo nhóm, thâm hụt `.vn`) trong một cây thư mục riêng, để không đè
lên kết quả chính. Vì tin phishing của phép chia theo thời gian đều có ngày, việc bỏ bản trùng gần như không làm đổi kết quả; bài báo cáo điều đó ở giới hạn (ix).

In [ ]:
def bare_host(urls: pd.Series) -> pd.Series:
    return (urls.astype(str).str.lower().str.replace(r"^[a-z][a-z0-9+.-]*://", "", regex=True)
            .str.replace(r"^www\.", "", regex=True))


def dedup_corpus():
    u = pd.read_csv(os.path.join(ROOT, "data", "processed", "dataset_url.csv"), low_memory=False)
    c = pd.read_csv(os.path.join(ROOT, "data", "processed", "vn_compphish.csv"), low_memory=False)
    s = u["collected_at"].astype(str).str.strip().str.slice(0, 10)
    dated = (pd.to_datetime(s, format="%d/%m/%Y", errors="coerce")
             .fillna(pd.to_datetime(s, format="%Y-%m-%d", errors="coerce"))).notna()
    host = u["url_norm"].astype(str).str.lower().str.replace(r"^[a-z][a-z0-9+.-]*://", "", regex=True)
    k = pd.DataFrame({"i": np.arange(len(u)), "bare": bare_host(u["url_norm"]),
                      "dated": dated, "www": host.str.startswith("www.")})
    k = k.sort_values(["bare", "dated", "www", "i"], ascending=[True, False, True, True])
    keep = np.sort(k.drop_duplicates("bare")["i"].to_numpy())
    return u.iloc[keep].reset_index(drop=True), c.iloc[keep].reset_index(drop=True)


DD = WORK / "dedup_tree"
(DD / "data/processed/p6").mkdir(parents=True, exist_ok=True)
ud, cd = dedup_corpus()
print(f"{len(ud):,} rows after keeping one row per host")
ud.to_csv(DD / "data/processed/dataset_url.csv", index=False)
cd.to_csv(DD / "data/processed/vn_compphish.csv", index=False)
(DD / "data/processed/brand_tokens.json").write_bytes((WORK / "data/processed/brand_tokens.json").read_bytes())

# The experiments write to paths fixed when their constants were defined; point them at the dedup tree for this run only
REDIRECT = ["STRATA_OUT", "AUC_OUT", "THR_OUT", "LEAK_OUT", "LOCUS_OUT", "GT_OUT_CSV", "PROC", "GROUPTHR_CSV", "BRANDS"]
saved_globals = {k: globals()[k] for k in REDIRECT}
try:
    for k in REDIRECT:
        globals()[k] = str(saved_globals[k]).replace(str(WORK), str(DD))
    os.chdir(DD)
    run_suffix_blindspot(skip_shap=True)
    run_group_threshold()
    run_vn_deficit()
finally:
    globals().update(saved_globals)
    os.chdir(WORK)
for f in ("p6_suffix_strata.csv", "p6_group_threshold.csv", "p6_vn_deficit.csv"):
    check_file("12 tier + dedup", DD / "data/processed/p6" / f, f"dedup_audit/{f}")
report("12 tier + dedup")

## Bước 13: Tổng hợp

Bảng dưới gom mọi phép kiểm theo bước. Mỗi dòng so một bảng kết quả (CSV) từng byte với bản kỳ vọng.

Riêng phần chạy lại trên corpus một-dòng-mỗi-URL, bảng đo theo tầng đuôi được chạy không kèm cột SHAP, đúng như bản mà bài dùng.

In [ ]:
tbl = pd.DataFrame(CHECKS)
per_step = tbl.groupby("step", sort=False)["match"].agg(checks="size", matched="sum")
per_step["mismatched"] = per_step["checks"] - per_step["matched"]
display(per_step)
print(f"total: {int(tbl['match'].sum())}/{len(tbl)} checks match")
bad = tbl[~tbl["match"]]
if len(bad):
    display(bad)